# GeoNUSAF — Section 3 figures

The six figures that use your data, on their own. Everything else in the paper is
already built by the other notebook; nothing here touches those files.

| key | figure |
|---|---|
| `1` | Fig. 1 — study area, locator and three tiles |
| `6` | Fig. 6 — the reliability weight field of Eq. (1) |
| `9` | Fig. 9 — qualitative comparison, both models run from your checkpoints |
| `11` | Fig. 11 — synthetic pairs and the scorer read-back |
| `A1` | Fig. A.1 — **merged**: (a) the pitch, (b) the scale-recovery budget |
| `GA` | the graphical abstract |

**How to use.** Set `CHOICE` and your image slots in the control panel, then Run All.
`run("9,11")` rebuilds a subset at any time. Everything lands in `figs/`, with one
`figs/figure_register.csv` and a zip you can download.

**What changed**

- **Part A of the control panel is one manual slot per image, for all six figures**, and
  nothing else in the notebook has to be edited. A slot takes a tile stem (`"image264"`),
  a full path, or `""` to let the notebook choose. Row labels are yours too.
- **Augmented corpora resolve again.** Files are matched on the tile index *and* whatever
  follows it, so `image217`, `image217hf` and `image217vf` are three tiles rather than one
  ambiguous index that no slot could be filled from. Automatic choices skip the flipped
  copies; `EXCLUDE_AUGMENTED = False` lets them back in.
- Fig. 1's locator uses Nepal's 2020 national boundary, and the panel under it is now the
  study area itself: your `.shp` boundary over satellite imagery fetched once and cached.
  `LOCATOR_IMAGE` still overrides it with a picture of your own. No invented city labels.
- Fig. 6 is recomputed the way training computes it: on the 512 cache, whole tile, with
  your `csa_weight` parameters, then cropped for display. The previous version cropped
  an image and a mask of different sizes with the same box, which is why the annotation
  did not match the photograph beside it.
- Fig. 9's proposed-model column now works: the TC-SegFormer module is your own class
  from `5__tc-segformer`, so `best.pt` loads strictly.
- Fig. 11 reads `genimage{i}.png` as **BGR** (the Stage-3 card says they are written
  with `cv2.imwrite`) and `genmask{i}.png` as index codes 0–5 with 255 ignore, and
  scores each pair with the B0 scorer named on that card.
- Fig. A.1 is one single-column figure, (a) above (b), sub-labels at the bottom.


## 0 · Control panel

In [ ]:
# =====================================================================================
#   CONTROL PANEL  -  the only cell you normally touch.  Set it, then Run All.
#   The build happens in the last cell.
#
#   PART A is one manual slot per image, for all six figures. Leave a slot as "" and
#   the notebook picks for you exactly as before; write anything into it and yours
#   wins. A slot takes a tile stem ("image264"), a file name ("image264.png") or a
#   full path ("/kaggle/input/.../whatever.png") - all three resolve.
# =====================================================================================

CHOICE = "all"     # "all" · "1" · "6" · "9" · "11" · "A1" · "GA" · or "1,9,11"

# =============================== PART A - MANUAL SLOTS ===============================

# ---------------------------------------------------------------------------- Fig. 1
# Three rows. Leave a mask "" and the annotation matching that image is used; leave an
# overlay "" and the 50 % blend is drawn here, or give it your own CVAT screenshot.
FIG1_IMAGES   = ["", "", ""]
FIG1_MASKS    = ["", "", ""]
FIG1_OVERLAYS = ["", "", ""]
FIG1_LABELS   = ["dense historic core", "river corridor", "cultivated fringe"]

# The panel under the Nepal map. "" builds the satellite view of the study-area
# boundary itself (section 3); a path puts your own picture there instead.
LOCATOR_IMAGE   = ""
LOCATOR_CAPTION = "approximate extent of the\n202.4 km$^2$ annotated corpus"

# ---------------------------------------------------------------------------- Fig. 6
FIG6_TILE  = ""           # one stem used for both panels (a) and (b)
FIG6_IMAGE = ""           # or set the two sides separately; these win over FIG6_TILE
FIG6_MASK  = ""
FIG6_CROP  = None         # (x0, y0, x1, y1) in 512-cache pixels; None = chosen for you

# ---------------------------------------------------------------------------- Fig. 9
# The two prediction columns are computed from your checkpoints, never pasted.
FIG9_IMAGES = ["", "", ""]
FIG9_MASKS  = ["", "", ""]
FIG9_LABELS = ["highest-scoring scene", "intermediate scene", "lowest-scoring scene"]

# --------------------------------------------------------------------------- Fig. 11
# An index is the i in genimage{i}.png / genmask{i}.png. Full paths also accepted.
FIG11_INDICES   = ["", "", ""]
FIG11_GENIMAGES = ["", "", ""]
FIG11_GENMASKS  = ["", "", ""]
FIG11_LABELS    = ["high fidelity", "mid fidelity", "low fidelity, river-bearing"]

# -------------------------------------------------------------------------- Fig. A.1
FIGA1_IMAGE = ""          # "" = image264, the tile holding the pitch

# ------------------------------------------------------------------ graphical abstract
GA_IMAGE = ""             # the tile shown in the left panel
GA_MASK  = ""             # its annotation; "" = the matching mask of GA_IMAGE

# ============================== PART B - EVERYTHING ELSE =============================

# ---- where the data is. "auto" scans /kaggle/input and finds the folders itself.
IMG_DIR   = "auto"
MASK_DIR  = "auto"
FAKE_ROOT = "auto"            # the "GeoNUSAF fake pairs block fold 1" dataset
FAKE_SET  = "fake_pairs_R1"   # fake_pairs_R1 | fake_pairs_R2 | fake_pairs_R3

# How an image is matched to its annotation. None = same file name first, then the
# trailing index together with whatever follows it (image217vf <-> mask217vf, and
# image217 stays distinct from image217hf). Give a function if your names need one:
#   MASK_FOR_IMAGE = lambda p: p.replace("/images/", "/masks/").replace("image", "mask")
MASK_FOR_IMAGE = None

# Flipped copies (…hf, …vf) are the same ground as the tile they came from, so the
# automatic choice ignores them and shows originals. False lets it use them too.
EXCLUDE_AUGMENTED = True

USE_DISPLAY_PALETTE = True    # False renders the released mask colours instead
SEED = 42

# ------------------------------------------------- Fig. 1, the study-area satellite view
# Only used when LOCATOR_IMAGE is "". The boundary comes from your shapefile; the
# imagery is fetched once from a public XYZ tile service and cached in figs/_cache.
VALLEY_SHP      = "auto"      # "auto" finds a .shp in the inputs · a path · "" = the
                              #  ring embedded in section 3 (kathmandu_shapefile.shp)
VALLEY_BASEMAP  = "esri"      # "esri" (needs internet) · "off" = outline only
VALLEY_TILE_URL = ("https://server.arcgisonline.com/ArcGIS/rest/services/"
                   "World_Imagery/MapServer/tile/{z}/{y}/{x}")
VALLEY_CREDIT   = "Imagery: Esri, Maxar, Earthstar Geographics"
                              # Esri's own full string adds ", and the GIS User
                              # Community"; use it if a reviewer asks for it
VALLEY_PAD      = 0.10        # margin around the boundary, as a fraction of its size
VALLEY_DIM      = 0.62        # how far everything outside the boundary fades to white
VALLEY_WIDTH_PX = 1400        # long side of the fetched mosaic; sets the zoom level
VALLEY_OUTLINE  = "#B2182B"
VALLEY_SCALEBAR = True

# -------------------------------------------------------------------------- Fig. A.1
PITCH_AUTO   = True           # find and rotate the pitch automatically
FIGA1_CROP   = None           # (x0, y0, x1, y1) in tile pixels; overrides the auto crop
FIGA1_ANGLE  = None           # degrees; overrides the automatic rotation
GSD_M_PER_PX = 0.3016         # the recovered scale, used to draw the four baselines
LINE_COLOUR  = "#FFFFFF"      # over turf and track, white with a dark halo reads best
HALO_COLOUR  = "#101010"

# -------------------------------------------------------------------------- models
# Fig. 9 needs both checkpoints, Fig. 11 needs the B0 scorer named on the Stage-3 card.
# Kaggle: Session options -> Internet -> On.
INFER = "auto"                # "auto" | "off"
HF_B0 = "Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold1"
HF_TC = "sugam24/geonusaf-tcsegformer-block-fold1"
FOLD  = 1
MAX_SCORE_SCENES = 60         # scenes scored when ranking Fig. 9 automatically
MAX_SCORE_PAIRS  = 80         # generated pairs scored when ranking Fig. 11

# ------------------------------------------------------------------ assembled rows
# The builders read these; they are just PART A in the shape the builders expect.
def _slot(seq, i):
    return seq[i] if i < len(seq) else ""

FIG1_ROWS = [dict(label=_slot(FIG1_LABELS, r), image=_slot(FIG1_IMAGES, r),
                  mask=_slot(FIG1_MASKS, r), overlay=_slot(FIG1_OVERLAYS, r))
             for r in range(3)]
FIG9_ROWS = [dict(label=_slot(FIG9_LABELS, r), image=_slot(FIG9_IMAGES, r),
                  mask=_slot(FIG9_MASKS, r)) for r in range(3)]
FIG11_ROWS = [dict(label=_slot(FIG11_LABELS, r), index=_slot(FIG11_INDICES, r),
                   image=_slot(FIG11_GENIMAGES, r), mask=_slot(FIG11_GENMASKS, r))
              for r in range(3)]
FIGA1_TILE = FIGA1_IMAGE or "image264"

_manual = sum(1 for v in (FIG1_IMAGES + FIG1_MASKS + FIG1_OVERLAYS + FIG9_IMAGES
                          + FIG9_MASKS + FIG11_INDICES + FIG11_GENIMAGES
                          + FIG11_GENMASKS + [FIG6_TILE, FIG6_IMAGE, FIG6_MASK,
                                              FIGA1_IMAGE, GA_IMAGE, GA_MASK,
                                              LOCATOR_IMAGE]) if v)
print("control panel set · CHOICE =", CHOICE, "·", _manual, "manual slot(s) filled")


## 1 · Setup, fonts and house style

Identical to the main notebook: Times/serif stack, 7 pt floor, Type 42 fonts,
constrained layout, sub-labels placed after layout is resolved.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from matplotlib.patches import Patch, FancyBboxPatch, FancyArrowPatch, Circle, Rectangle
from matplotlib.lines import Line2D
import matplotlib.font_manager as fm

OUT = Path("figs"); OUT.mkdir(exist_ok=True)
SAVE_EPS = False     # Elsevier accepts EPS or PDF; PDF is the safer of the two
SAVE_PNG = True      # 600 dpi previews for checking. Submit the PDFs, not these.

MM = 1 / 25.4
W1, W15, W2 = 90 * MM, 140 * MM, 190 * MM     # Elsevier: single / 1.5 / double column

# ---- Times. Liberation Serif and Nimbus Roman are metrically identical to Times New Roman,
# ---- so on Kaggle or Linux the output is indistinguishable in print.
SERIF = ["Times New Roman", "Nimbus Roman", "Liberation Serif", "TeX Gyre Termes",
         "FreeSerif", "DejaVu Serif"]
have = {f.name for f in fm.fontManager.ttflist}
resolved = next((f for f in SERIF if f in have), "DejaVu Serif")
print("serif family in use:", resolved)
if resolved == "DejaVu Serif":
    print("  Times not found. On Kaggle run:  !apt-get -qq install -y fonts-liberation")
    print("  then restart the kernel. DejaVu Serif is a safe stand-in meanwhile.")

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "pdf.fonttype": 42,          # embed TrueType. Type 3 fails Elsevier preflight.
    "ps.fonttype": 42,
    "font.family": "serif",
    "font.serif": SERIF,
    "mathtext.fontset": "stix",  # Times-compatible math
    "font.size": 7,              # Elsevier floor is 7 pt at final size
    "axes.labelsize": 7.5,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "legend.frameon": False,
    "legend.handlelength": 1.3,
    "legend.handletextpad": 0.5,
    "legend.columnspacing": 1.4,
    "legend.borderpad": 0.0,
    "axes.titlesize": 7.5,
    "axes.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.linewidth": 0.4,
    "grid.color": "0.88",
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.4,
    "ytick.major.size": 2.4,
    "hatch.linewidth": 0.5,
})

SUBLABEL_PT = 8      # size of (a) (b) (c)
# Panels stacked one per row share a single x, so (a) and (b) sit on one vertical
# line: a wide image and a chart carrying long tick labels do not have the same
# centre, and centring each label on its own panel puts them at two different x.
#   "union" (default) · "mean" · a figure fraction such as 0.5 · None = per panel
SUBLABEL_STACK_X = "union"
NOTE = dict(fontsize=5.6, color="0.55", style="italic")   # rare in-figure note style

REGISTER = []

def top_legend(fig, handles, labels, ncol, loc="outside upper center"):
    # Legend above the plot, outside the axes, so it can never sit on the data.
    return fig.legend(handles=handles, labels=labels, loc=loc, ncol=ncol,
                      frameon=False, fontsize=7, borderaxespad=0.15)

def finish(fig, name, figure_no, section, where, float_env, caption, axes=None, hspace=0.12):
    # axes: pass the panel axes in order to get (a) (b) (c) below them, on one baseline.
    # An entry may itself be a list of axes that share one sub-label (Fig. 1b).
    if axes:
        h_in = fig.get_size_inches()[1]
        reserve = (5.6 * MM) / h_in            # strip of figure height kept free for the labels
        eng = fig.get_layout_engine()

        from matplotlib.transforms import Bbox

        def boxes_now():
            fig.draw_without_rendering()
            r = fig.canvas.get_renderer()
            out = []
            for item in axes:
                group_ = item if isinstance(item, (list, tuple)) else [item]
                bs = [a.get_tightbbox(r).transformed(fig.transFigure.inverted())
                      for a in group_]
                out.append(Bbox.from_extents(min(b.x0 for b in bs), min(b.y0 for b in bs),
                                             max(b.x1 for b in bs), max(b.y1 for b in bs)))
            return out

        eng.set(rect=(0, reserve, 1, 1 - reserve))   # (left, bottom, width, height)
        bx = boxes_now()

        def cluster(bs):
            # group panels into rows by vertical centre, top row first
            idx = sorted(range(len(bs)), key=lambda i: -0.5 * (bs[i].y0 + bs[i].y1))
            rows, cur = [], [idx[0]]
            for i in idx[1:]:
                c0 = 0.5 * (bs[cur[-1]].y0 + bs[cur[-1]].y1)
                c1 = 0.5 * (bs[i].y0 + bs[i].y1)
                if abs(c0 - c1) < 0.06:
                    cur.append(i)
                else:
                    rows.append(cur); cur = [i]
            rows.append(cur)
            return rows

        rows = cluster(bx)
        if len(rows) > 1:            # stacked panels need a gap to hold the label
            eng.set(hspace=hspace)
            bx = boxes_now(); rows = cluster(bx)
        stack = len(rows) > 1 and all(len(r) == 1 for r in rows)
        xcom = None
        if stack and SUBLABEL_STACK_X is not None:
            if isinstance(SUBLABEL_STACK_X, (int, float)):
                xcom = float(SUBLABEL_STACK_X)
            elif SUBLABEL_STACK_X == "mean":
                xcom = float(np.mean([0.5 * (bx[r[0]].x0 + bx[r[0]].x1) for r in rows]))
            else:
                xcom = 0.5 * (min(b.x0 for b in bx) + max(b.x1 for b in bx))
        for row in rows:
            ybase = min(bx[i].y0 for i in row) - 0.30 * reserve
            for i in row:
                fig.text(xcom if xcom is not None else 0.5 * (bx[i].x0 + bx[i].x1),
                         ybase, "(" + "abcdefgh"[i] + ")",
                         ha="center", va="top", fontsize=SUBLABEL_PT)
        fig.set_layout_engine("none")

    fig.savefig(OUT / (name + ".pdf"))
    if SAVE_EPS:
        fig.savefig(OUT / (name + ".eps"))
    if SAVE_PNG:
        fig.savefig(OUT / (name + ".png"))
    w, h = fig.get_size_inches()
    size = str(round(w / MM)) + " x " + str(round(h / MM)) + " mm"
    print(name + ".pdf   " + size + "   " + float_env)
    print("   " + figure_no + "  ->  " + section + ", " + where)
    REGISTER.append(dict(file=name + ".pdf", figure=figure_no, section=section,
                         insertion_point=where, float_env=float_env, size=size,
                         caption=caption))

print("ready. output ->", OUT.resolve())

## 2 · Palette, reading and decoding

The class order and the released mask colours are the ones in your training
notebook's `BGR_TO_CLASS`. Masks are decoded by measuring the file, so colour masks,
byte-reversed masks and index-coded masks all work.

In [ ]:
import random, csv, json
from PIL import Image

rng = random.Random(SEED)
RASTER = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
CACHE = OUT / "_cache"; CACHE.mkdir(parents=True, exist_ok=True)

# ---- classes in the order the checkpoints use (model cards, and BGR_TO_CLASS in the
# ---- training notebook). Everything in this notebook indexes classes this way.
CLASSES = ["Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural"]

# The released masks are written with cv2, so the file's RGB values are these:
RELEASED = {"Residential": "#000080", "Road": "#008000", "River": "#800000",
            "Forest": "#808000", "UnusedLand": "#008080", "Agricultural": "#800080",
            "Unannotated": "#000000"}
DISPLAY = {"Residential": "#C0392B", "Road": "#4D4D4D", "River": "#1F6FB4",
           "Forest": "#14532D", "UnusedLand": "#C2A477", "Agricultural": "#E8B62C",
           "Unannotated": "#FFFFFF"}
CLR = DISPLAY if USE_DISPLAY_PALETTE else RELEASED
print("palette:", "display" if USE_DISPLAY_PALETTE else "released")


def hex2rgb(h):
    return np.array([int(h[i:i + 2], 16) for i in (1, 3, 5)], np.uint8)


def _key(rgb):
    return (int(rgb[0]) << 16) | (int(rgb[1]) << 8) | int(rgb[2])


import re as _re

_DIGIT_RUN = _re.compile(r"\d+")


def listdir(d):
    if not d or not Path(str(d)).is_dir():
        return []
    return sorted(str(p) for p in Path(d).iterdir() if p.suffix.lower() in RASTER)


def stem(p):
    return Path(p).stem


def split_stem(s):
    """image217vf -> ("image", 217, "vf").

    The LAST run of digits is the tile index; whatever follows it is the variant
    suffix. That suffix is what the augmented copies of a tile differ by, so keeping
    it makes image217, image217hf and image217vf three separate tiles instead of one
    index that three files answer to. Zero padding is absorbed by the int, so mask021
    and image21 still meet.
    """
    s = str(s)
    runs = list(_DIGIT_RUN.finditer(s))
    if not runs:
        return s.lower(), None, ""
    g = runs[-1]
    return s[:g.start()].lower(), int(g.group()), s[g.end():].lower()


def _num(s):
    """The tile index of a file stem: image264 -> 264, mask021 -> 21, else None."""
    return split_stem(s)[1]


def _digits(s):
    n = _num(s)
    return "" if n is None else str(n)


def tile_key(s):
    """(index, variant suffix) - what two files must share to be the same tile."""
    pre, n, suf = split_stem(s)
    return None if n is None else (n, suf)


def is_augmented(s):
    """True for a flipped or otherwise suffixed copy: image217hf, image217vf."""
    return split_stem(s)[2] != ""


# ------------------------------------------------------------------ find the dataset
def _candidates(roots, min_files=4, max_depth=6):
    seen = {}
    skip = (str(OUT.resolve()), "_cache", "_preds", ".git", "__pycache__")
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.rglob("*"):
            try:
                if not p.is_dir() or any(t in str(p.resolve()) for t in skip):
                    continue
                if len(p.relative_to(root).parts) > max_depth:
                    continue
                n = sum(1 for q in p.iterdir() if q.suffix.lower() in RASTER)
                if n >= min_files:
                    seen[str(p)] = n
            except Exception:
                continue
    return seen


def _few_colours(paths, k=3, limit=24):
    u = set()
    for p in paths[:k]:
        a = thumb(p, 96)
        if a is None:
            continue
        u |= set(map(tuple, np.unique(a.reshape(-1, 3), axis=0).tolist()))
    return len(u) <= limit


def thumb(path, n=192):
    try:
        im = Image.open(path)
        im = im.convert("RGB") if im.mode not in ("RGB", "L") else im
        if n and max(im.size) > n:
            im = im.resize((n, n), Image.NEAREST)
        a = np.array(im)
        return a[..., :3] if a.ndim == 3 else np.dstack([a] * 3)
    except Exception:
        return None


def discover():
    cands = _candidates([Path("/kaggle/input"), Path("input"), Path("data"), Path(".")])
    info = {}
    for d, n in cands.items():
        ps = listdir(d)
        low = d.lower()
        info[d] = dict(n=n, paths=ps,
                       mask=(_few_colours(ps) or any(t in low for t in
                             ("mask", "label", "annot", "gt", "seg"))),
                       fake=any(t in low for t in ("gen", "fake", "synth", "stage3")))
    def best(is_mask, is_fake):
        pool = [(v["n"], d) for d, v in info.items()
                if v["mask"] == is_mask and v["fake"] == is_fake]
        return max(pool)[1] if pool else ""
    img = best(False, False)
    msk = ""
    if img:
        want = {_digits(stem(p)) for p in info[img]["paths"][:60]}
        sc = [(len(want & {_digits(stem(q)) for q in v["paths"][:60]}), v["n"], d)
              for d, v in info.items() if v["mask"] and not v["fake"]]
        msk = max(sc)[2] if sc else ""
    fake_img = best(False, True)
    root = ""
    if fake_img:
        p = Path(fake_img)
        root = str(p.parent.parent if FAKE_SET in str(p) else p.parent)
    return img, msk, root, info


_img, _msk, _root, _info = discover() if "auto" in (IMG_DIR, MASK_DIR, FAKE_ROOT) \
    else ("", "", "", {})
IMG_DIR = _img if IMG_DIR == "auto" else IMG_DIR
MASK_DIR = _msk if MASK_DIR == "auto" else MASK_DIR
FAKE_ROOT = _root if FAKE_ROOT == "auto" else FAKE_ROOT

REAL_IMGS, REAL_MASKS = listdir(IMG_DIR), listdir(MASK_DIR)


def _fake_dir(root, which):
    if not root:
        return ""
    p = Path(root)
    for cand in (p / which, p, p / which / "images"):
        if cand.is_dir() and any(q.name.startswith("gen") for q in cand.iterdir()):
            return str(cand)
    for q in p.rglob(which):
        if q.is_dir():
            return str(q)
    return ""


FAKE_DIR = _fake_dir(FAKE_ROOT, FAKE_SET)
FAKE_FILES = listdir(FAKE_DIR)
FAKE_IMGS = [p for p in FAKE_FILES if "genimage" in Path(p).name]
FAKE_MASKS = [p for p in FAKE_FILES if "genmask" in Path(p).name]
if not FAKE_IMGS and FAKE_DIR:                     # images and masks in sibling folders
    FAKE_IMGS = [p for p in listdir(str(Path(FAKE_DIR) / "genimages")) ]
    FAKE_MASKS = [p for p in listdir(str(Path(FAKE_DIR) / "genmasks"))]

print("images      :", len(REAL_IMGS), " <-", IMG_DIR or "NOT FOUND")
print("masks       :", len(REAL_MASKS), " <-", MASK_DIR or "NOT FOUND")
print("fake pairs  :", len(FAKE_IMGS), "images /", len(FAKE_MASKS), "masks  <-",
      FAKE_DIR or "NOT FOUND")
if not REAL_IMGS:
    print("\nNo tiles found. Attach the dataset, or set IMG_DIR / MASK_DIR by hand.")


# ------------------------------------------------------------------ reading
def load_rgb(path, bgr=False):
    """Read an image as RGB. bgr=True for files written with cv2.imwrite."""
    try:
        a = np.array(Image.open(path).convert("RGB"))
    except Exception:
        return None
    return a[..., ::-1].copy() if bgr else a


_COLOUR_LUTS = []
for _pal in (RELEASED, DISPLAY):
    for _rev in (False, True):
        _COLOUR_LUTS.append({_key(hex2rgb(v)[::-1] if _rev else hex2rgb(v)):
                             (CLASSES.index(k) if k in CLASSES else 255)
                             for k, v in _pal.items()})


def to_index(mask):
    """Any mask -> HxW class indices (255 = unannotated). Measured, not assumed."""
    if mask is None:
        return None, 0.0
    a = mask if mask.ndim == 3 else np.dstack([mask] * 3)
    grey = np.array_equal(a[..., 0], a[..., 1]) and np.array_equal(a[..., 1], a[..., 2])
    if grey and len(np.unique(a[..., 0])) <= 8:                 # index-coded {0..5} + 255
        g = a[..., 0].astype(np.int16)
        out = np.where(g < len(CLASSES), g, 255).astype(np.int16)
        return out, float((out != 255).mean())
    keys = (a[..., 0].astype(np.int32) << 16) | (a[..., 1].astype(np.int32) << 8) | a[..., 2]
    best, cov = None, -1.0
    for lut in _COLOUR_LUTS:
        out = np.full(keys.shape, 255, np.int16)
        for k, ci in lut.items():
            out[keys == k] = ci
        c = float((out != 255).mean())
        if c > cov:
            best, cov = out, c
    return best, cov


def paint(idx):
    """class indices -> the figure palette."""
    out = np.zeros(idx.shape + (3,), np.uint8)
    out[:] = hex2rgb(CLR["Unannotated"])
    for k, c in enumerate(CLASSES):
        out[idx == k] = hex2rgb(CLR[c])
    return out


def load_mask(path):
    idx, cov = to_index(load_rgb(path))
    return None if idx is None else paint(idx)


def resize_to(a, hw, nearest=False):
    if a is None or a.shape[:2] == tuple(hw):
        return a
    im = Image.fromarray(a if a.ndim == 3 else a.astype(np.uint8))
    return np.array(im.resize((hw[1], hw[0]),
                              Image.NEAREST if nearest else Image.BILINEAR))


def overlay(img, mask, alpha=0.45):
    if img is None or mask is None:
        return None
    mask = resize_to(mask, img.shape[:2], nearest=True)
    return (img.astype(np.float32) * (1 - alpha)
            + mask.astype(np.float32) * alpha).astype(np.uint8)


def crop(a, box):
    if a is None or box is None:
        return a
    x0, y0, x1, y1 = box
    return a[y0:y1, x0:x1]


# ------------------------------------------------------------------ resolving slots
def find_file(spec, pool):
    """A stem, a path, or "" -> a path from pool (or None).

    Matching is on the tile index AND the suffix after it, so image217vf answers to
    mask217vf and to nothing else. Index alone is what used to make every augmented
    corpus ambiguous: three files (image217, image217hf, image217vf) all claimed
    index 217, the slot was refused, and the figure came out with no image.
    """
    if not spec:
        return None
    spec = str(spec)
    if Path(spec).is_file():
        return spec
    # "image217.png" is a file NAME and "/some/old/dir/image217.png" is a path that has
    # moved. Neither is a stem and neither exists here, so both are reduced to the stem
    # the pool is keyed on. Without this the ".png" is read as the variant suffix, the
    # tile key becomes (217, ".png"), nothing carries it, and a slot that names a real
    # file looks like it has no counterpart.
    if Path(spec).suffix.lower() in RASTER or "/" in spec or "\\" in spec:
        if str(Path(spec).parent) not in (".", ""):
            print("   NOTE: %s is not on disk; matching on the name %s."
                  % (spec, Path(spec).name))
        spec = Path(spec).stem
    for p in pool:                                     # 1. the same file stem
        if stem(p) == spec:
            return p
    low = spec.lower()
    for p in pool:                                     # 2. the same stem, any case
        if stem(p).lower() == low:
            return p
    k = tile_key(spec)
    if k is None:
        return None
    hits = [p for p in pool if tile_key(stem(p)) == k]  # 3. index + variant suffix
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:                                  # same index and same suffix:
        hits.sort(key=lambda p: (len(stem(p)), stem(p)))   # only the prefix differs
        print("   NOTE: %d files are index %d%s; using %s."
              % (len(hits), k[0], " suffix '%s'" % k[1] if k[1] else "", stem(hits[0])))
        return hits[0]
    hits = [p for p in pool if _num(stem(p)) == k[0]]  # 4. index alone, last resort
    if len(hits) == 1:
        return hits[0]
    base = [p for p in hits if not is_augmented(stem(p))]
    if len(base) == 1:                                 # the unflipped tile of that index
        print("   NOTE: nothing matches %s exactly; using %s." % (spec, stem(base[0])))
        return base[0]
    if hits:
        print("   NO COUNTERPART for %s: %d files carry index %d (%s) and none carries"
              % (spec, len(hits), k[0], ", ".join(stem(h) for h in hits[:4])))
        print("   its suffix. Set the slot to a full path, or set EXCLUDE_AUGMENTED = True.")
    return None


def mask_for(image_path):
    """The annotation belonging to one image, by explicit rule then by tile key."""
    if image_path is None:
        return None
    if callable(MASK_FOR_IMAGE):
        p = MASK_FOR_IMAGE(image_path)
        return p if (p and Path(p).is_file()) else None
    for p in REAL_MASKS:                       # identical file names in the two folders
        if Path(p).name == Path(image_path).name:
            return p
    return find_file(stem(image_path), REAL_MASKS)


def image_for(mask_path):
    """The tile belonging to one annotation - the same rule, read backwards."""
    if mask_path is None:
        return None
    for p in REAL_IMGS:
        if Path(p).name == Path(mask_path).name:
            return p
    return find_file(stem(mask_path), REAL_IMGS)


def green_index(rgb):
    a = rgb.astype(np.float32)
    return a[..., 1] - 0.5 * (a[..., 0] + a[..., 2])


def pair_ok(image_path, mask_path, verbose=True):
    """Do this image and this annotation describe the same ground?

    Vegetation is greener than built-up. It is a crude test, but a mismatched pair
    fails it almost every time, and no correct pair in a land-use corpus should.
    Returns True, False, or None when the tile cannot decide (too few pixels).
    """
    img = thumb(image_path, 192); mk = thumb(mask_path, 192)
    if img is None or mk is None:
        return None
    idx, cov = to_index(mk)
    if idx is None or cov < 0.2 or idx.shape != img.shape[:2]:
        return None
    gi = green_index(img)
    veg = np.isin(idx, [CLASSES.index("Forest"), CLASSES.index("Agricultural")])
    built = np.isin(idx, [CLASSES.index("Residential"), CLASSES.index("Road")])
    if veg.mean() < 0.03 or built.mean() < 0.03:
        return None
    d = float(gi[veg].mean() - gi[built].mean())
    if verbose and d <= 0:
        print("   PAIRING WARNING: in %s the pixels labelled vegetation are LESS green"
              % Path(image_path).name)
        print("   than the pixels labelled built-up (%.1f). This image and this mask are"
              % d)
        print("   probably not the same tile.")
    return d > 0


def pairing_report(sample=24):
    """Run the check across the corpus before any figure is drawn."""
    if not (REAL_IMGS and REAL_MASKS):
        return
    print("\npairing check")
    print("   images <-", IMG_DIR)
    print("   masks  <-", MASK_DIR)
    if Path(IMG_DIR).parent != Path(MASK_DIR).parent:
        print("   NOTE: these two folders are not siblings. If they come from different")
        print("   datasets they may use different numbering - that is the single most")
        print("   likely way to get an image and a mask that do not match.")
    idxs = np.linspace(0, len(REAL_IMGS) - 1, min(sample, len(REAL_IMGS))).astype(int)
    ok = bad = skip = nomask = 0
    shown = 0
    for i in idxs:
        ip = REAL_IMGS[i]; mp = mask_for(ip)
        if mp is None:
            nomask += 1; continue
        r = pair_ok(ip, mp, verbose=False)
        if shown < 4:
            print("   %-22s <-> %-22s %s" % (Path(ip).name, Path(mp).name,
                                             {True: "ok", False: "MISMATCH",
                                              None: "undecidable"}[r]))
            shown += 1
        ok += (r is True); bad += (r is False); skip += (r is None)
    tested = ok + bad
    print("   %d of %d sampled tiles pass, %d fail, %d undecidable, %d have no mask"
          % (ok, tested, bad, skip, nomask))
    if tested and ok / tested < 0.7:
        print("   THE PAIRING IS WRONG. Do not use any figure built from it. Set IMG_DIR")
        print("   and MASK_DIR to the two folders of ONE dataset, or give MASK_FOR_IMAGE")
        print("   a rule that maps an image path to its annotation.")
    return ok, bad


def show(ax, arr, note="", frame="0.6"):
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_linewidth(0.5); sp.set_color(frame)
    if arr is None:
        ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes, fc="#F4F4F4",
                               ec="#B2182B", lw=0.8))
        if note:
            ax.text(0.5, 0.5, note, transform=ax.transAxes, ha="center", va="center",
                    fontsize=6.2, color="#B2182B")
        return
    ax.imshow(arr, interpolation="nearest")


# the legend keeps the reading order used by the other 21 figures, so the whole
# figure set looks like one set; the colours per class are identical either way
LEGEND_ORDER = ["Residential", "Agricultural", "UnusedLand", "Forest", "Road", "River"]


def class_legend(fig, ncol=7):
    return top_legend(fig, [Patch(fc=CLR[c], ec="0.4", lw=0.3) for c in LEGEND_ORDER]
                      + [Patch(fc=CLR["Unannotated"], ec="0.4", lw=0.3)],
                      LEGEND_ORDER + ["unannotated"], ncol=ncol)


# ------------------------------------------------------------------ light census
CENSUS_PATH = CACHE / "tile_census.csv"


def build_census(force=False, n=192):
    if not REAL_MASKS:
        return {}
    sig = str(len(REAL_MASKS)) + "|" + str(n)
    if CENSUS_PATH.exists() and not force:
        try:
            rows = list(csv.DictReader(open(CENSUS_PATH)))
            if rows and rows[0].get("_sig") == sig:
                return {r["stem"]: {c: float(r[c]) for c in CLASSES} for r in rows}
        except Exception:
            pass
    print("counting class pixels in", len(REAL_MASKS), "masks (cached after this) ...")
    stats = {}
    for i, p in enumerate(REAL_MASKS):
        idx, cov = to_index(thumb(p, n))
        if idx is None or cov < 0.2:
            continue
        stats[stem(p)] = {c: float((idx == k).mean()) for k, c in enumerate(CLASSES)}
        if (i + 1) % 100 == 0:
            print("   %d / %d" % (i + 1, len(REAL_MASKS)))
    with open(CENSUS_PATH, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["stem", "_sig"] + CLASSES); w.writeheader()
        for s, d in stats.items():
            r = dict(d); r["stem"] = s; r["_sig"] = sig; w.writerow(r)
    return stats


TILE_STATS = build_census()


def auto_stems():
    """The stems automatic selection may choose from.

    A flipped copy is the same ground as the tile it came from, so by default the
    figures are built from originals and the ...hf / ...vf tiles are left to training.
    """
    ss = list(TILE_STATS)
    if EXCLUDE_AUGMENTED:
        base = [s for s in ss if not is_augmented(s)]
        if base:
            return base
    return ss


pairing_report()
if TILE_STATS:
    mean = {c: float(np.mean([v[c] for v in TILE_STATS.values()])) for c in CLASSES}
    tot = sum(mean.values()) or 1.0
    print("measured class share:", ", ".join("%s %.1f%%" % (c[:4], 100 * mean[c] / tot)
                                             for c in CLASSES))
print("io ready")

## 3 · The locator map

In [ ]:
# ---------------------------------------------------------------------------------
#  Fig. 1(a) - the locator.
#
#  The outline is Nepal's national boundary as published by the Government of Nepal
#  (the 2020 map, which shows Limpiyadhura, Kalapani and Lipulekh), taken from the
#  National Geoportal shapefiles, simplified to 900 vertices and embedded here so the
#  cell needs no internet. That boundary is disputed with India, which is exactly why
#  the caption carries the sentence JAG requires of any map figure: map lines delineate
#  study areas and do not necessarily depict accepted national boundaries.
#
#  Under it sits whatever image you put in LOCATOR_IMAGE.
# ---------------------------------------------------------------------------------

NEPAL = [
    (80.606,30.467), (80.615,30.455), (80.627,30.458), (80.661,30.439), (80.692,30.410),
    (80.711,30.413), (80.736,30.397), (80.789,30.350), (80.806,30.320), (80.866,30.310),
    (80.887,30.296), (80.910,30.300), (80.938,30.260), (80.985,30.269), (81.024,30.246),
    (81.036,30.234), (81.031,30.220), (81.047,30.210), (81.035,30.197), (81.085,30.148),
    (81.083,30.101), (81.100,30.100), (81.110,30.088), (81.093,30.054), (81.126,30.020),
    (81.165,30.010), (81.186,30.011), (81.197,30.023), (81.219,30.007), (81.247,30.011),
    (81.255,30.024), (81.239,30.032), (81.274,30.045), (81.291,30.074), (81.292,30.113),
    (81.260,30.151), (81.307,30.164), (81.319,30.152), (81.334,30.153), (81.352,30.178),
    (81.397,30.207), (81.394,30.251), (81.410,30.261), (81.418,30.282), (81.401,30.282),
    (81.404,30.293), (81.425,30.305), (81.398,30.323), (81.408,30.353), (81.397,30.368),
    (81.422,30.371), (81.431,30.383), (81.402,30.396), (81.410,30.421), (81.447,30.416),
    (81.491,30.381), (81.554,30.370), (81.553,30.403), (81.561,30.407), (81.563,30.428),
    (81.609,30.412), (81.624,30.424), (81.620,30.446), (81.636,30.445), (81.729,30.411),
    (81.734,30.398), (81.753,30.389), (81.811,30.387), (81.828,30.375), (81.870,30.377),
    (81.884,30.361), (81.916,30.350), (81.941,30.348), (81.951,30.359), (81.964,30.351),
    (81.959,30.341), (81.990,30.321), (82.014,30.329), (82.018,30.344), (82.058,30.333),
    (82.069,30.357), (82.087,30.359), (82.113,30.345), (82.130,30.302), (82.111,30.277),
    (82.120,30.260), (82.102,30.237), (82.111,30.226), (82.129,30.226), (82.141,30.199),
    (82.161,30.189), (82.186,30.191), (82.184,30.161), (82.207,30.154), (82.201,30.136),
    (82.173,30.122), (82.188,30.111), (82.168,30.081), (82.174,30.065), (82.233,30.065),
    (82.249,30.075), (82.283,30.061), (82.300,30.038), (82.336,30.047), (82.365,30.010),
    (82.380,30.011), (82.378,30.026), (82.397,30.024), (82.434,29.989), (82.478,29.974),
    (82.476,29.965), (82.501,29.946), (82.532,29.971), (82.551,29.963), (82.565,29.955),
    (82.569,29.932), (82.587,29.921), (82.604,29.889), (82.644,29.871), (82.623,29.837),
    (82.642,29.834), (82.699,29.855), (82.713,29.839), (82.729,29.835), (82.739,29.814),
    (82.712,29.793), (82.696,29.791), (82.692,29.763), (82.726,29.771), (82.754,29.766),
    (82.769,29.742), (82.765,29.731), (82.820,29.721), (82.817,29.705), (82.828,29.690),
    (82.863,29.685), (82.893,29.701), (82.937,29.705), (82.950,29.692), (82.943,29.674),
    (82.958,29.663), (83.003,29.665), (83.011,29.673), (83.023,29.651), (83.035,29.654),
    (83.045,29.631), (83.067,29.626), (83.071,29.610), (83.088,29.603), (83.092,29.612),
    (83.119,29.616), (83.122,29.629), (83.134,29.618), (83.152,29.624), (83.167,29.616),
    (83.161,29.596), (83.182,29.593), (83.199,29.611), (83.220,29.605), (83.217,29.589),
    (83.234,29.578), (83.280,29.569), (83.284,29.558), (83.257,29.557), (83.269,29.507),
    (83.284,29.499), (83.318,29.509), (83.349,29.493), (83.344,29.466), (83.368,29.450),
    (83.373,29.429), (83.415,29.422), (83.428,29.402), (83.415,29.367), (83.447,29.340),
    (83.446,29.300), (83.467,29.275), (83.489,29.280), (83.494,29.265), (83.517,29.256),
    (83.525,29.238), (83.521,29.218), (83.541,29.203), (83.574,29.202), (83.581,29.179),
    (83.612,29.181), (83.633,29.160), (83.643,29.161), (83.665,29.183), (83.658,29.204),
    (83.686,29.213), (83.713,29.245), (83.764,29.240), (83.799,29.250), (83.797,29.277),
    (83.815,29.302), (83.850,29.298), (83.859,29.310), (83.907,29.326), (83.945,29.316),
    (83.965,29.331), (83.978,29.329), (84.003,29.312), (83.992,29.298), (84.029,29.286),
    (84.044,29.301), (84.071,29.285), (84.092,29.294), (84.122,29.280), (84.115,29.263),
    (84.118,29.244), (84.143,29.236), (84.195,29.244), (84.205,29.237), (84.202,29.214),
    (84.180,29.207), (84.164,29.185), (84.177,29.131), (84.203,29.124), (84.204,29.104),
    (84.188,29.081), (84.192,29.047), (84.224,29.053), (84.250,29.037), (84.244,28.976),
    (84.223,28.916), (84.228,28.894), (84.256,28.890), (84.268,28.898), (84.290,28.877),
    (84.326,28.863), (84.345,28.873), (84.359,28.859), (84.377,28.864), (84.411,28.853),
    (84.403,28.830), (84.421,28.831), (84.437,28.821), (84.434,28.801), (84.448,28.783),
    (84.439,28.769), (84.463,28.751), (84.494,28.735), (84.517,28.748), (84.527,28.742),
    (84.559,28.752), (84.574,28.735), (84.629,28.736), (84.646,28.725), (84.671,28.683),
    (84.707,28.671), (84.697,28.636), (84.739,28.617), (84.783,28.609), (84.859,28.570),
    (84.903,28.593), (84.947,28.595), (84.951,28.580), (84.986,28.591), (84.995,28.600),
    (84.991,28.613), (85.061,28.682), (85.086,28.671), (85.117,28.687), (85.159,28.643),
    (85.190,28.642), (85.187,28.632), (85.200,28.621), (85.176,28.597), (85.194,28.569),
    (85.189,28.534), (85.168,28.532), (85.155,28.490), (85.121,28.485), (85.102,28.457),
    (85.117,28.443), (85.118,28.409), (85.129,28.392), (85.110,28.346), (85.123,28.336),
    (85.184,28.324), (85.205,28.340), (85.244,28.318), (85.257,28.294), (85.273,28.297),
    (85.275,28.288), (85.343,28.304), (85.351,28.288), (85.380,28.278), (85.414,28.308),
    (85.418,28.327), (85.460,28.337), (85.509,28.334), (85.564,28.310), (85.603,28.304),
    (85.611,28.257), (85.663,28.303), (85.656,28.317), (85.668,28.348), (85.689,28.347),
    (85.686,28.382), (85.714,28.386), (85.727,28.373), (85.722,28.359), (85.740,28.319),
    (85.735,28.291), (85.751,28.238), (85.779,28.208), (85.802,28.196), (85.818,28.203),
    (85.831,28.185), (85.851,28.184), (85.854,28.167), (85.874,28.144), (85.872,28.124),
    (85.900,28.109), (85.903,28.054), (85.979,27.996), (85.951,27.941), (86.003,27.911),
    (86.016,27.915), (86.067,27.899), (86.069,27.911), (86.126,27.927), (86.126,27.941),
    (86.110,27.956), (86.106,27.984), (86.082,28.020), (86.097,28.062), (86.085,28.091),
    (86.094,28.098), (86.120,28.091), (86.133,28.100), (86.144,28.126), (86.180,28.143),
    (86.176,28.168), (86.189,28.174), (86.200,28.138), (86.222,28.111), (86.225,28.096),
    (86.210,28.094), (86.207,28.084), (86.228,28.031), (86.229,27.981), (86.240,27.973),
    (86.291,27.978), (86.321,27.948), (86.338,27.965), (86.350,27.937), (86.384,27.938),
    (86.413,27.906), (86.445,27.905), (86.472,27.930), (86.472,27.946), (86.519,27.957),
    (86.524,28.019), (86.538,28.050), (86.551,28.051), (86.568,28.069), (86.564,28.105),
    (86.577,28.113), (86.607,28.100), (86.611,28.076), (86.637,28.070), (86.692,28.108),
    (86.741,28.104), (86.767,28.074), (86.749,28.050), (86.755,28.038), (86.776,28.038),
    (86.793,28.019), (86.838,28.015), (86.855,28.027), (86.870,28.025), (86.892,27.997),
    (86.928,27.988), (86.937,27.962), (86.985,27.949), (87.018,27.957), (87.042,27.950),
    (87.056,27.926), (87.081,27.915), (87.091,27.882), (87.110,27.875), (87.118,27.840),
    (87.174,27.821), (87.228,27.819), (87.241,27.839), (87.263,27.851), (87.306,27.843),
    (87.320,27.826), (87.360,27.828), (87.376,27.845), (87.421,27.862), (87.434,27.842),
    (87.406,27.834), (87.452,27.822), (87.493,27.846), (87.539,27.843), (87.565,27.867),
    (87.589,27.853), (87.592,27.820), (87.614,27.810), (87.638,27.834), (87.668,27.836),
    (87.668,27.807), (87.701,27.812), (87.728,27.805), (87.781,27.866), (87.787,27.899),
    (87.816,27.921), (87.827,27.920), (87.834,27.953), (87.860,27.948), (87.857,27.929),
    (87.868,27.911), (87.935,27.918), (87.987,27.886), (88.033,27.906), (88.067,27.886),
    (88.087,27.895), (88.098,27.867), (88.138,27.881), (88.198,27.854), (88.204,27.837),
    (88.176,27.820), (88.200,27.791), (88.184,27.776), (88.181,27.747), (88.159,27.736),
    (88.147,27.665), (88.123,27.645), (88.116,27.610), (88.085,27.585), (88.078,27.572),
    (88.084,27.565), (88.062,27.535), (88.070,27.526), (88.064,27.512), (88.046,27.496),
    (88.047,27.478), (88.070,27.458), (88.071,27.438), (88.083,27.430), (88.053,27.413),
    (88.044,27.372), (88.069,27.336), (88.044,27.304), (88.047,27.293), (88.034,27.286),
    (88.033,27.250), (88.015,27.216), (88.022,27.171), (87.990,27.119), (87.994,27.105),
    (88.024,27.087), (88.019,27.073), (88.040,27.036), (88.062,27.029), (88.080,27.035),
    (88.095,27.004), (88.138,26.985), (88.123,26.950), (88.149,26.921), (88.139,26.898),
    (88.177,26.859), (88.170,26.837), (88.191,26.771), (88.189,26.738), (88.170,26.699),
    (88.163,26.636), (88.145,26.625), (88.132,26.601), (88.133,26.573), (88.115,26.571),
    (88.116,26.558), (88.108,26.561), (88.114,26.545), (88.101,26.537), (88.106,26.519),
    (88.093,26.507), (88.101,26.505), (88.110,26.475), (88.094,26.463), (88.098,26.439),
    (88.074,26.408), (88.037,26.392), (88.033,26.365), (88.010,26.361), (87.995,26.369),
    (87.994,26.392), (87.929,26.418), (87.919,26.432), (87.927,26.445), (87.901,26.448),
    (87.912,26.457), (87.891,26.486), (87.888,26.476), (87.861,26.463), (87.864,26.449),
    (87.851,26.436), (87.831,26.440), (87.832,26.450), (87.791,26.469), (87.776,26.450),
    (87.779,26.422), (87.770,26.411), (87.742,26.417), (87.736,26.407), (87.714,26.427),
    (87.680,26.435), (87.681,26.415), (87.653,26.406), (87.654,26.393), (87.615,26.390),
    (87.607,26.380), (87.590,26.382), (87.590,26.392), (87.555,26.405), (87.550,26.418),
    (87.469,26.440), (87.371,26.408), (87.359,26.388), (87.361,26.361), (87.343,26.347),
    (87.315,26.371), (87.268,26.374), (87.269,26.405), (87.259,26.411), (87.231,26.413),
    (87.221,26.403), (87.214,26.413), (87.188,26.414), (87.192,26.405), (87.165,26.404),
    (87.094,26.450), (87.074,26.585), (87.047,26.587), (87.045,26.568), (87.018,26.532),
    (86.935,26.516), (86.932,26.489), (86.894,26.486), (86.896,26.462), (86.863,26.457),
    (86.836,26.439), (86.769,26.459), (86.746,26.435), (86.749,26.428), (86.733,26.422),
    (86.723,26.435), (86.702,26.437), (86.696,26.451), (86.636,26.464), (86.623,26.484),
    (86.573,26.496), (86.544,26.538), (86.500,26.542), (86.458,26.568), (86.397,26.585),
    (86.399,26.593), (86.343,26.606), (86.347,26.616), (86.336,26.619), (86.255,26.617),
    (86.237,26.590), (86.220,26.588), (86.198,26.594), (86.195,26.614), (86.138,26.606),
    (86.141,26.629), (86.029,26.666), (86.012,26.654), (85.978,26.658), (85.963,26.651),
    (85.950,26.613), (85.931,26.616), (85.894,26.594), (85.890,26.583), (85.851,26.569),
    (85.860,26.590), (85.853,26.608), (85.833,26.611), (85.833,26.601), (85.820,26.602),
    (85.804,26.624), (85.772,26.630), (85.736,26.650), (85.739,26.664), (85.728,26.670),
    (85.726,26.687), (85.737,26.796), (85.723,26.820), (85.645,26.853), (85.637,26.872),
    (85.626,26.873), (85.587,26.847), (85.578,26.861), (85.568,26.841), (85.545,26.838),
    (85.522,26.814), (85.500,26.812), (85.498,26.797), (85.482,26.802), (85.480,26.792),
    (85.457,26.782), (85.410,26.791), (85.377,26.760), (85.358,26.760), (85.337,26.741),
    (85.242,26.762), (85.214,26.758), (85.179,26.809), (85.193,26.831), (85.192,26.870),
    (85.113,26.872), (85.089,26.867), (85.089,26.853), (85.024,26.855), (85.031,26.875),
    (85.062,26.887), (85.006,26.895), (84.995,26.903), (85.004,26.903), (85.002,26.912),
    (84.970,26.916), (84.980,26.940), (84.965,26.960), (84.894,26.970), (84.858,26.986),
    (84.857,27.008), (84.828,27.021), (84.814,27.021), (84.796,26.996), (84.775,27.018),
    (84.759,27.003), (84.645,27.046), (84.648,27.075), (84.677,27.096), (84.672,27.110),
    (84.689,27.158), (84.677,27.171), (84.687,27.180), (84.683,27.200), (84.693,27.215),
    (84.625,27.336), (84.479,27.362), (84.472,27.355), (84.443,27.370), (84.394,27.368),
    (84.303,27.390), (84.294,27.385), (84.279,27.395), (84.280,27.415), (84.256,27.452),
    (84.234,27.456), (84.222,27.440), (84.211,27.442), (84.208,27.469), (84.151,27.486),
    (84.148,27.519), (84.126,27.511), (84.111,27.522), (84.100,27.517), (84.098,27.489),
    (84.073,27.482), (84.054,27.443), (84.029,27.433), (83.998,27.446), (83.980,27.439),
    (83.933,27.449), (83.871,27.429), (83.875,27.392), (83.907,27.390), (83.908,27.383),
    (83.890,27.358), (83.865,27.346), (83.832,27.370), (83.617,27.469), (83.391,27.479),
    (83.404,27.457), (83.386,27.436), (83.412,27.413), (83.387,27.390), (83.390,27.376),
    (83.345,27.334), (83.297,27.333), (83.268,27.365), (83.274,27.383), (83.220,27.417),
    (83.190,27.454), (83.036,27.449), (82.956,27.468), (82.931,27.501), (82.737,27.502),
    (82.739,27.518), (82.751,27.523), (82.740,27.552), (82.749,27.555), (82.758,27.588),
    (82.737,27.609), (82.738,27.643), (82.714,27.665), (82.720,27.687), (82.708,27.703),
    (82.710,27.722), (82.683,27.714), (82.666,27.722), (82.621,27.702), (82.607,27.706),
    (82.543,27.686), (82.518,27.690), (82.475,27.676), (82.451,27.679), (82.404,27.703),
    (82.369,27.743), (82.306,27.772), (82.210,27.843), (82.156,27.866), (82.152,27.858),
    (82.125,27.865), (82.073,27.923), (81.970,27.929), (81.927,27.876), (81.928,27.860),
    (81.887,27.857), (81.835,27.898), (81.807,27.903), (81.700,27.988), (81.626,28.001),
    (81.480,28.082), (81.486,28.118), (81.452,28.141), (81.448,28.161), (81.376,28.177),
    (81.369,28.165), (81.380,28.155), (81.373,28.142), (81.320,28.133), (81.320,28.145),
    (81.330,28.148), (81.304,28.166), (81.323,28.169), (81.322,28.197), (81.234,28.289),
    (81.236,28.325), (81.212,28.361), (81.087,28.383), (81.037,28.413), (81.034,28.399),
    (81.018,28.416), (81.036,28.428), (81.031,28.438), (81.021,28.432), (81.015,28.438),
    (81.019,28.452), (81.001,28.454), (80.991,28.442), (80.980,28.442), (80.981,28.434),
    (80.967,28.445), (80.982,28.455), (80.953,28.447), (80.953,28.456), (80.934,28.459),
    (80.944,28.468), (80.932,28.458), (80.916,28.468), (80.917,28.458), (80.903,28.486),
    (80.910,28.498), (80.878,28.510), (80.863,28.504), (80.867,28.490), (80.846,28.495),
    (80.817,28.521), (80.791,28.524), (80.791,28.543), (80.771,28.550), (80.770,28.566),
    (80.740,28.574), (80.740,28.566), (80.726,28.575), (80.716,28.568), (80.692,28.587),
    (80.697,28.603), (80.679,28.604), (80.681,28.620), (80.669,28.642), (80.653,28.634),
    (80.643,28.643), (80.613,28.639), (80.608,28.652), (80.591,28.648), (80.599,28.656),
    (80.582,28.661), (80.592,28.680), (80.542,28.691), (80.508,28.671), (80.523,28.577),
    (80.532,28.572), (80.524,28.553), (80.483,28.585), (80.470,28.583), (80.466,28.615),
    (80.453,28.629), (80.417,28.639), (80.373,28.629), (80.328,28.666), (80.319,28.700),
    (80.279,28.710), (80.258,28.729), (80.267,28.743), (80.258,28.754), (80.218,28.755),
    (80.120,28.829), (80.077,28.824), (80.066,28.837), (80.060,28.916), (80.102,28.986),
    (80.120,28.982), (80.128,29.006), (80.137,29.007), (80.126,29.013), (80.138,29.059),
    (80.130,29.061), (80.146,29.103), (80.185,29.137), (80.235,29.118), (80.273,29.142),
    (80.260,29.187), (80.242,29.202), (80.250,29.221), (80.289,29.195), (80.299,29.205),
    (80.292,29.232), (80.302,29.242), (80.319,29.315), (80.298,29.324), (80.282,29.346),
    (80.273,29.365), (80.281,29.379), (80.276,29.393), (80.243,29.412), (80.245,29.444),
    (80.276,29.450), (80.271,29.455), (80.278,29.459), (80.282,29.451), (80.304,29.453),
    (80.288,29.477), (80.301,29.491), (80.315,29.485), (80.316,29.499), (80.351,29.519),
    (80.360,29.533), (80.344,29.540), (80.344,29.553), (80.366,29.557), (80.365,29.564),
    (80.381,29.561), (80.381,29.571), (80.410,29.595), (80.407,29.613), (80.422,29.639),
    (80.416,29.654), (80.386,29.674), (80.367,29.748), (80.399,29.759), (80.409,29.794),
    (80.436,29.806), (80.495,29.796), (80.509,29.823), (80.554,29.850), (80.574,29.895),
    (80.576,29.923), (80.597,29.953), (80.632,29.965), (80.650,29.956), (80.680,29.959),
    (80.721,30.000), (80.749,30.007), (80.763,30.050), (80.781,30.057), (80.808,30.090),
    (80.842,30.099), (80.846,30.116), (80.880,30.129), (80.873,30.161), (80.763,30.279),
    (80.758,30.306), (80.669,30.343), (80.656,30.379), (80.640,30.389), (80.613,30.444),
    (80.595,30.448), (80.551,30.430), (80.550,30.443), (80.567,30.459), (80.606,30.467)
]
VALLEY_LONLAT = (85.37, 27.71)     # marker only, not a tile footprint
MAP_SOURCE = "National boundary: Government of Nepal (2020)"


# ---------------------------------------------------------------------------------
#  The panel under the map: the study area itself, on satellite imagery.
#
#  The boundary is read straight out of your shapefile - the geometry record only, so
#  the .shp on its own is enough and no .dbf, no .shx and no GIS package is needed.
#  The imagery beneath it is one mosaic of XYZ tiles, fetched once and cached in
#  figs/_cache; everything outside the boundary is faded back so the shape still reads
#  at 40 mm wide. Set LOCATOR_IMAGE to put your own picture there instead, and
#  VALLEY_BASEMAP = "off" to draw the outline with no imagery at all.
# ---------------------------------------------------------------------------------
import math, struct, json as _json, hashlib, urllib.request

#  kathmandu_shapefile.shp, WGS84 lon/lat, one ring of 52 vertices, embedded so the
#  panel still builds when the file is not attached.
KTM_RING = [
    (85.331444,27.741796), (85.332730,27.741668), (85.340416,27.738339), (85.345442,27.729657), (85.344751,27.724109),
    (85.346437,27.718657), (85.344873,27.712589), (85.343297,27.707531), (85.347170,27.705630), (85.351313,27.705883),
    (85.352185,27.702336), (85.354593,27.700443), (85.355019,27.695383), (85.353176,27.691706), (85.350915,27.687635),
    (85.349919,27.685983), (85.349458,27.680821), (85.349460,27.678160), (85.347603,27.676639), (85.342335,27.672674),
    (85.339761,27.671277), (85.336037,27.668879), (85.332597,27.667001), (85.330163,27.663725), (85.327013,27.660112),
    (85.323871,27.658371), (85.322159,27.658251), (85.319876,27.660145), (85.314881,27.663929), (85.310439,27.665883),
    (85.308434,27.666505), (85.302124,27.674194), (85.302112,27.683432), (85.294396,27.685348), (85.287531,27.688010),
    (85.282521,27.691557), (85.281686,27.697013), (85.281658,27.702833), (85.282249,27.705496), (85.283126,27.708917),
    (85.282766,27.714479), (85.284176,27.718020), (85.296698,27.724620), (85.303834,27.725394), (85.304840,27.727671),
    (85.305699,27.730583), (85.305701,27.733367), (85.305992,27.735387), (85.318725,27.735006), (85.326161,27.738528),
    (85.329590,27.740795), (85.331444,27.741796)
]


def read_shp_rings(path):
    """Every polygon or polyline ring in a .shp file, as [(lon, lat), ...].

    Shapefile geometry lives entirely in the .shp; the .dbf holds attributes and the
    .shx is an index, so neither is needed to draw an outline.
    """
    b = Path(path).read_bytes()
    if len(b) < 100 or struct.unpack(">i", b[:4])[0] != 9994:
        return []
    rings, off = [], 100
    while off + 8 <= len(b):
        rl = struct.unpack(">i", b[off + 4:off + 8])[0]
        end = off + 8 + 2 * rl
        p = off + 8
        st = struct.unpack("<i", b[p:p + 4])[0]; p += 4
        if st in (3, 5, 13, 15, 23, 25):                 # (poly)line / polygon, Z, M
            p += 32                                      # the per-record bounding box
            nparts, npts = struct.unpack("<ii", b[p:p + 8]); p += 8
            parts = struct.unpack("<%di" % nparts, b[p:p + 4 * nparts]); p += 4 * nparts
            xy = struct.unpack("<%dd" % (2 * npts), b[p:p + 16 * npts])
            for i, s0 in enumerate(parts):
                s1 = parts[i + 1] if i + 1 < nparts else npts
                rings.append([(xy[2 * j], xy[2 * j + 1]) for j in range(s0, s1)])
        off = end
    return rings


def find_shapefile():
    for root in (Path("/kaggle/input"), Path("."), Path("input"), Path("data")):
        if not root.exists():
            continue
        for q in sorted(root.rglob("*.shp")):
            return str(q)
    return ""


def valley_rings():
    """The study-area boundary: your shapefile if there is one, else the embedded ring."""
    src = find_shapefile() if VALLEY_SHP == "auto" else VALLEY_SHP
    if src:
        try:
            rs = [r for r in read_shp_rings(src) if len(r) >= 4]
            lons = [p[0] for r in rs for p in r]; lats = [p[1] for r in rs for p in r]
            if rs and max(map(abs, lons)) <= 180 and max(map(abs, lats)) <= 90:
                print("   boundary <-", src, "·", len(rs), "ring(s),",
                      sum(len(r) for r in rs), "vertices")
                return rs
            if rs:
                print("   %s is not in degrees (projected CRS); using the embedded ring"
                      % Path(src).name)
        except Exception as e:
            print("   could not read", src, "-", e)
    return [KTM_RING]


# ------------------------------------------------------------------ the tile mosaic
def _webmerc(lon, lat, z):
    n = 2.0 ** z
    s = min(max(math.sin(math.radians(lat)), -0.9999), 0.9999)
    return ((lon + 180.0) / 360.0 * n,
            (0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)) * n)


def _get_tile(url, z, x, y, tries=2):
    u = url.format(z=z, x=x, y=y)
    for _ in range(tries):
        try:
            rq = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(rq, timeout=25) as fh:
                import io
                return Image.open(io.BytesIO(fh.read())).convert("RGB")
        except Exception:
            continue
    return None


def fetch_mosaic(bbox, want_px=1400, url="", max_tiles=180):
    """(w, s, e, n) in degrees -> (RGB array, z, tile origin, crop offset), or None."""
    w, s, e, n = bbox
    z = 10
    for zz in range(10, 19):
        x0, y0 = _webmerc(w, n, zz); x1, y1 = _webmerc(e, s, zz)
        if 256 * max(x1 - x0, y1 - y0) > want_px * 1.35:
            break
        z = zz
    x0, y0 = _webmerc(w, n, z); x1, y1 = _webmerc(e, s, z)
    tx0, ty0, tx1, ty1 = int(x0), int(y0), int(x1), int(y1)
    ntiles = (tx1 - tx0 + 1) * (ty1 - ty0 + 1)
    if ntiles > max_tiles:
        print("   %d tiles at zoom %d is too many; lower VALLEY_WIDTH_PX" % (ntiles, z))
        return None
    print("   fetching %d imagery tiles at zoom %d ..." % (ntiles, z))
    canvas = Image.new("RGB", ((tx1 - tx0 + 1) * 256, (ty1 - ty0 + 1) * 256), (240, 240, 240))
    got = 0
    for tx in range(tx0, tx1 + 1):
        for ty in range(ty0, ty1 + 1):
            im = _get_tile(url, z, tx, ty)
            if im is not None:
                canvas.paste(im, ((tx - tx0) * 256, (ty - ty0) * 256)); got += 1
    if got < 0.6 * ntiles:
        print("   only %d of %d tiles arrived - no internet? Session options -> Internet -> On."
              % (got, ntiles))
        return None
    left, top = int(round((x0 - tx0) * 256)), int(round((y0 - ty0) * 256))
    right, bot = int(round((x1 - tx0) * 256)), int(round((y1 - ty0) * 256))
    return np.array(canvas.crop((left, top, right, bot))), z, (tx0, ty0), (left, top)


_VALLEY = {}


def valley_view():
    """The finished panel: imagery, the boundary in pixel coordinates, and the scale."""
    if _VALLEY:
        return _VALLEY.get("view")
    rings = valley_rings()
    lons = [p[0] for r in rings for p in r]; lats = [p[1] for r in rings for p in r]
    w, e, s, n = min(lons), max(lons), min(lats), max(lats)
    padx, pady = VALLEY_PAD * (e - w), VALLEY_PAD * (n - s)
    bbox = (w - padx, s - pady, e + padx, n + pady)
    lat0 = 0.5 * (s + n)

    tag = hashlib.md5(("%s|%.5f,%.5f,%.5f,%.5f|%d"
                       % ((VALLEY_TILE_URL,) + bbox + (VALLEY_WIDTH_PX,))).encode()).hexdigest()[:10]
    png, side = CACHE / ("valley_%s.png" % tag), CACHE / ("valley_%s.json" % tag)
    got = None
    if png.exists() and side.exists():
        try:
            meta = _json.loads(side.read_text())
            got = (np.array(Image.open(png).convert("RGB")), meta["z"],
                   tuple(meta["origin"]), tuple(meta["crop"]))
            print("   imagery from cache ·", png.name)
        except Exception:
            got = None
    if got is None and VALLEY_BASEMAP != "off":
        got = fetch_mosaic(bbox, VALLEY_WIDTH_PX, VALLEY_TILE_URL)
        if got is not None:
            Image.fromarray(got[0]).save(png)
            side.write_text(_json.dumps(dict(z=got[1], origin=list(got[2]), crop=list(got[3]))))

    if got is not None:
        rgb, z, (tx0, ty0), (left, top) = got

        def to_px(lon, lat):
            X, Y = _webmerc(lon, lat, z)
            return ((X - tx0) * 256 - left, (Y - ty0) * 256 - top)
        m_per_px = 156543.03392 * math.cos(math.radians(lat0)) / (2.0 ** z)
    else:                                   # no imagery: a plain panel, outline only
        H = 900
        W = max(int(round(H * (bbox[2] - bbox[0]) * math.cos(math.radians(lat0))
                          / (bbox[3] - bbox[1]))), 200)
        rgb = np.full((H, W, 3), 244, np.uint8)

        def to_px(lon, lat):
            return ((lon - bbox[0]) / (bbox[2] - bbox[0]) * W,
                    (bbox[3] - lat) / (bbox[3] - bbox[1]) * H)
        m_per_px = 111320.0 * (bbox[3] - bbox[1]) / H
        if VALLEY_BASEMAP != "off":
            print("   no imagery - drawing the boundary alone. Put your own picture in")
            print("   LOCATOR_IMAGE, or turn the session's internet on.")

    poly = [np.array([to_px(lo, la) for lo, la in r]) for r in rings]

    if got is not None and VALLEY_DIM > 0:          # fade everything outside the area
        from PIL import ImageDraw
        m = Image.new("L", (rgb.shape[1], rgb.shape[0]), 0)
        d = ImageDraw.Draw(m)
        for q in poly:
            d.polygon([tuple(v) for v in q], fill=255)
        inside = (np.array(m) > 127)[..., None]
        faded = (rgb.astype(np.float32) * (1 - VALLEY_DIM) + 255.0 * VALLEY_DIM)
        rgb = np.where(inside, rgb, faded.astype(np.uint8))

    _VALLEY["view"] = dict(rgb=rgb, poly=poly, m_per_px=m_per_px,
                           imagery=got is not None)
    return _VALLEY["view"]


def _scalebar(ax, arr, m_per_px):
    H, W = arr.shape[:2]
    target = 0.30 * W * m_per_px
    nice = min([250, 500, 1000, 2000, 5000], key=lambda v: abs(v - target))
    L = nice / m_per_px
    x0, y0 = 0.06 * W, 0.955 * H
    ax.plot([x0, x0 + L], [y0, y0], color="white", lw=2.2, solid_capstyle="butt",
            zorder=5, path_effects=None)
    ax.plot([x0, x0 + L], [y0, y0], color="#202020", lw=1.0, solid_capstyle="butt",
            zorder=6)
    ax.text(x0 + L / 2, y0 - 0.018 * H,
            ("%d m" % nice) if nice < 1000 else ("%g km" % (nice / 1000.0)),
            ha="center", va="bottom", fontsize=5.4, color="#202020", zorder=6,
            bbox=dict(fc="white", ec="none", alpha=0.72, pad=0.8))


def draw_locator(ax):
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    for sp in ax.spines.values():
        sp.set_linewidth(0.5); sp.set_color("0.6")

    a1 = ax.inset_axes([0.03, 0.755, 0.94, 0.215])
    xs = [p[0] for p in NEPAL]; ys = [p[1] for p in NEPAL]
    a1.fill(xs, ys, fc="#EDEDED", ec="#5A5A5A", lw=0.6, zorder=1)
    a1.plot([VALLEY_LONLAT[0]], [VALLEY_LONLAT[1]], marker="s", ms=3.0,
            mfc="#B2182B", mec="#B2182B", zorder=3)
    a1.annotate("Kathmandu Valley", VALLEY_LONLAT,
                xytext=(VALLEY_LONLAT[0] + 0.7, VALLEY_LONLAT[1] + 1.7),
                fontsize=6.0, color="#B2182B", ha="left",
                arrowprops=dict(arrowstyle="-", lw=0.6, color="#B2182B",
                                shrinkA=0, shrinkB=1.5))
    a1.text(80.6, 26.7, "NEPAL", fontsize=6.4, color="#4D4D4D", style="italic")
    a1.set_xlim(79.7, 88.7); a1.set_ylim(25.9, 31.2)
    a1.set_aspect(1.0 / np.cos(np.deg2rad(28.3))); a1.axis("off")

    source = MAP_SOURCE
    a2 = ax.inset_axes([0.03, 0.215, 0.94, 0.50])
    img = load_rgb(LOCATOR_IMAGE) if (LOCATOR_IMAGE and Path(LOCATOR_IMAGE).is_file()) else None
    if img is None and LOCATOR_IMAGE:
        img = load_rgb(find_file(LOCATOR_IMAGE, REAL_IMGS) or "")
    if img is not None:                                   # your own picture
        show(a2, img)
        cap_y = 0.185
    else:
        v = valley_view()
        if v is None:
            a2.remove()                   # nothing to show: no empty box, no gap
            cap_y = 0.66
        else:
            arr = v["rgb"]
            a2.imshow(arr, interpolation="bilinear", zorder=0)
            for q in v["poly"]:
                a2.plot(np.r_[q[:, 0], q[0, 0]], np.r_[q[:, 1], q[0, 1]],
                        color=VALLEY_OUTLINE, lw=0.9, zorder=4)
            if VALLEY_SCALEBAR:
                _scalebar(a2, arr, v["m_per_px"])
            a2.set_xlim(0, arr.shape[1]); a2.set_ylim(arr.shape[0], 0)
            a2.set_xticks([]); a2.set_yticks([])
            a2.add_patch(Rectangle((0, 0), arr.shape[1] - 1, arr.shape[0] - 1,
                                   fc="none", ec="0.6", lw=0.5, zorder=7))
            for sp in a2.spines.values():
                sp.set_visible(False)
            if v["imagery"] and VALLEY_CREDIT:
                source = MAP_SOURCE + "\n" + VALLEY_CREDIT
            cap_y = 0.185

    ax.text(0.5, cap_y, LOCATOR_CAPTION, ha="center", va="top", fontsize=6.2,
            color="#333333")
    ax.text(0.5, 0.03, source, ha="center", va="bottom", fontsize=5.4,
            color="#8C8C8C", style="italic", linespacing=1.35)


print("locator ready ·", len(NEPAL), "outline vertices")


## 4 · Checkpoints

In [ ]:
# =====================================================================================
#  Checkpoints.  Fig. 9 needs both models, Fig. 11 needs the B0 scorer that the
#  Stage-3 card names as the layout-fidelity scorer.
#
#  The TC-SegFormer module below is your own class, copied from the training notebook
#  (5__tc-segformer, the STANDALONE cell), so the state dict loads strictly rather
#  than by guesswork. Preprocessing is yours too: BGR read, INTER_AREA to 512,
#  reverse to RGB, ImageNet normalisation.
#
#  Kaggle: Session options -> Internet -> On. With INFER = "off" nothing is downloaded
#  and the prediction panels say what is missing instead of showing something invented.
# =====================================================================================

PRED_STATUS = {"segformer-b0": "not attempted", "tc-segformer": "not attempted"}
MODELS = {}
_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
_STD = np.array([0.229, 0.224, 0.225], np.float32)


def _build_tc(C):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from transformers import SegformerConfig, SegformerForSemanticSegmentation

    class _Detail(nn.Module):
        def __init__(s, chs):
            super().__init__(); c1, c2, c3 = chs
            b = lambda i, o, st: nn.Sequential(nn.Conv2d(i, o, 3, st, 1, bias=False),
                                               nn.GroupNorm(min(8, o), o), nn.ReLU(inplace=True))
            s.net = nn.Sequential(b(3, c1, 2), b(c1, c2, 1), b(c2, c3, 1)); s.out_ch = c3
        def forward(s, x): return s.net(x)

    class _TC(nn.Module):
        def __init__(s, C, refine_name="classifier"):
            super().__init__()
            conf = SegformerConfig.from_pretrained(C["BACKBONE"])
            conf.num_labels = C["NUM_CLASSES"]
            base = SegformerForSemanticSegmentation.from_pretrained(
                C["BACKBONE"], config=conf, ignore_mismatched_sizes=True)
            s.encoder, s.head = base.segformer, base.decode_head
            s.use_detail = C["USE_DETAIL"]
            s._refine_name = refine_name
            if s.use_detail:
                s.proj = nn.Conv2d(conf.decoder_hidden_size, C["FUSE_CH"], 1)
                s.detail = _Detail(tuple(C["DETAIL_CH"]))
                s.gate = nn.Parameter(torch.tensor(0.1))
                s.fuse = nn.Sequential(
                    nn.Conv2d(C["FUSE_CH"] + s.detail.out_ch, C["FUSE_CH"], 3, 1, 1, bias=False),
                    nn.GroupNorm(8, C["FUSE_CH"]), nn.ReLU(inplace=True))
                setattr(s, refine_name, nn.Conv2d(C["FUSE_CH"], C["NUM_CLASSES"], 1))

        def forward(s, x):
            H, W = x.shape[-2:]
            hs = s.encoder(x, output_hidden_states=True, return_dict=True).hidden_states
            outs = []
            for h_, mlp in zip(hs, s.head.linear_c):
                hh, ww = h_.shape[2], h_.shape[3]
                t = mlp(h_).permute(0, 2, 1).reshape(h_.shape[0], -1, hh, ww)
                outs.append(F.interpolate(t, size=hs[0].shape[2:], mode="bilinear",
                                          align_corners=False))
            fused = s.head.dropout(s.head.activation(
                s.head.batch_norm(s.head.linear_fuse(torch.cat(outs[::-1], 1)))))
            logits = F.interpolate(s.head.classifier(fused), size=(H, W),
                                   mode="bilinear", align_corners=False)
            if not s.use_detail:
                return logits
            f = F.interpolate(s.proj(fused), size=(H // 2, W // 2), mode="bilinear",
                              align_corners=False)
            f = torch.cat([f, s.gate * s.detail(x)], 1)
            r = getattr(s, s._refine_name)(s.fuse(f))
            return logits + F.interpolate(r, size=(H, W), mode="bilinear", align_corners=False)

    return _TC, C


def load_tc(repo):
    """Your checkpoint holds model / cfg / metrics_full / epoch."""
    import torch
    from huggingface_hub import hf_hub_download
    ck = torch.load(hf_hub_download(repo, "best.pt"), map_location="cpu", weights_only=False)
    C = ck["cfg"] if isinstance(ck.get("cfg"), dict) else dict(
        BACKBONE="nvidia/segformer-b0-finetuned-ade-512-512", NUM_CLASSES=6,
        USE_DETAIL=True, DETAIL_CH=(8, 16, 32), FUSE_CH=64)
    sd = ck.get("model", ck.get("model_state", ck))
    sd = {k.replace("module.", ""): v for k, v in sd.items()}
    _TC, C = _build_tc(C)
    last = None
    for name in ("classifier", "refine"):      # the export cell and the training cell
        try:                                   # name the residual head differently
            net = _TC(C, name)
            net.load_state_dict(sd, strict=True)
            net.eval()
            note = "" if name == "classifier" else " (residual head named 'refine')"
            return net, "loaded from " + repo + note + \
                (" · val mIoU %.4f" % ck["metrics_full"]["mIoU"]
                 if isinstance(ck.get("metrics_full"), dict) else "")
        except Exception as e:
            last = e
    raise RuntimeError(str(last).split("\n")[0][:200])


def load_b0(repo):
    import torch
    from huggingface_hub import hf_hub_download
    from transformers import SegformerForSemanticSegmentation, SegformerConfig
    ck = torch.load(hf_hub_download(repo, "best.pt"), map_location="cpu", weights_only=False)
    cfg = ck.get("model_config")
    net = SegformerForSemanticSegmentation(SegformerConfig(**cfg)) if isinstance(cfg, dict) \
        else SegformerForSemanticSegmentation.from_pretrained(repo)
    sd = ck.get("model_state", ck.get("model", ck))
    sd = {k.replace("module.", ""): v for k, v in sd.items()}
    miss, unexp = net.load_state_dict(sd, strict=False)
    net.eval()
    return net, "loaded from " + repo + (" (%d missing, %d unexpected keys)"
                                         % (len(miss), len(unexp)) if miss or unexp else "")


if INFER != "off":
    for tag, repo, fn in (("segformer-b0", HF_B0, load_b0), ("tc-segformer", HF_TC, load_tc)):
        try:
            MODELS[tag], PRED_STATUS[tag] = fn(repo)
        except Exception as e:
            PRED_STATUS[tag] = "not loaded - " + str(e).split("\n")[0][:170]
else:
    PRED_STATUS = {k: "skipped (INFER = 'off')" for k in PRED_STATUS}

for k, v in PRED_STATUS.items():
    print("%-14s %s" % (k, v))
if INFER != "off" and len(MODELS) < 2:
    print("\nOn Kaggle this is nearly always the Internet switch: Session options ->")
    print("Internet -> On, then re-run this cell.")

_DEV = "cpu"
try:
    import torch as _t
    _DEV = "cuda" if _t.cuda.is_available() else "cpu"
    for _m in MODELS.values():
        _m.to(_DEV)
    if MODELS:
        print("running on", _DEV)
except Exception:
    pass


def predict(model, rgb, size=512):
    """rgb uint8 HxWx3 -> class indices at size x size."""
    import torch
    a = np.array(Image.fromarray(rgb).resize((size, size), Image.BILINEAR), np.float32) / 255.0
    x = torch.from_numpy(((a - _MEAN) / _STD).transpose(2, 0, 1))[None].to(_DEV)
    with torch.no_grad():
        out = model(x)
        logits = out.logits if hasattr(out, "logits") else out
        if logits.shape[-1] != size:
            import torch.nn.functional as F
            logits = F.interpolate(logits, size=(size, size), mode="bilinear",
                                   align_corners=False)
    return logits.float().argmax(1)[0].cpu().numpy().astype(np.int16)


def iou_mean(pred, gt):
    vals = []
    for k in range(len(CLASSES)):
        p, g = (pred == k), (gt == k)
        u = np.logical_or(p, g).sum()
        if u:
            vals.append(float(np.logical_and(p, g).sum()) / u)
    return float(np.mean(vals)) if vals else np.nan


def scorer():
    return MODELS.get("segformer-b0") or MODELS.get("tc-segformer")

## 5 · The pitch panel

In [ ]:
# ---------------------------------------------------------------------------------
#  Fig. A.1(a) - the pitch.
#
#  The stadium sits inside dense city, so the panel is cropped to the pitch itself and
#  rotated upright, exactly as the appendix describes. Turf is found by colour, the
#  long axis by principal components, and the four regulation baselines are drawn at
#  their true pixel length, GSD_M_PER_PX metres to the pixel.
# ---------------------------------------------------------------------------------
from scipy import ndimage as ndi
import matplotlib.patheffects as pe

PITCH_CTL = dict(gr_min=15, gb_min=-10, g_lo=45, g_hi=170, r_hi=130, open_px=15)


def find_pitch(rgb, ctl=PITCH_CTL):
    """Largest patch of turf; returns centre, long-axis angle (deg), length, width."""
    R, G, B = (rgb[..., i].astype(int) for i in range(3))
    turf = ((G - R > ctl["gr_min"]) & (G - B > ctl["gb_min"]) & (G > ctl["g_lo"])
            & (G < ctl["g_hi"]) & (R < ctl["r_hi"]))
    turf = ndi.binary_closing(turf, np.ones((9, 9)))
    turf = ndi.binary_opening(turf, np.ones((ctl["open_px"], ctl["open_px"])))
    lab, n = ndi.label(turf)
    if n == 0:
        return None
    sizes = ndi.sum(turf, lab, range(1, n + 1))
    ys, xs = np.nonzero(lab == int(np.argmax(sizes)) + 1)
    pts = np.stack([xs, ys], 1).astype(float)
    c = pts.mean(0)
    _, _, vt = np.linalg.svd(pts - c, full_matrices=False)
    ext = (pts - c) @ vt.T
    return dict(centre=c, angle=float(np.degrees(np.arctan2(vt[0, 1], vt[0, 0]))),
                length=float(np.ptp(ext[:, 0])), width=float(np.ptp(ext[:, 1])),
                area=float(sizes.max()))


def upright_pitch(rgb, angle=None, margin_x=0.45, margin_y=0.10):
    """Rotate so the pitch stands upright, then crop to it. Returns (crop, rect)."""
    p = find_pitch(rgb)
    if p is None:
        return None, None
    ang = p["angle"] if angle is None else angle
    half = int(0.85 * max(p["length"], p["width"]))
    cx, cy = p["centre"]
    x0, y0 = int(max(cx - half, 0)), int(max(cy - half, 0))
    x1, y1 = int(min(cx + half, rgb.shape[1])), int(min(cy + half, rgb.shape[0]))
    sub = rgb[y0:y1, x0:x1]
    rot = np.array(Image.fromarray(sub).rotate(ang - 90, resample=Image.BICUBIC,
                                               expand=True, fillcolor=(0, 0, 0)))
    q = find_pitch(rot)                       # re-measure: now axis aligned
    if q is None:
        return rot, None
    L = max(q["length"], q["width"]); Wd = min(q["length"], q["width"])
    mx, my = margin_x * Wd, margin_y * L       # wide side margins fill the column width
    cx2, cy2 = q["centre"]
    bx0 = int(max(cx2 - Wd / 2 - mx, 0)); bx1 = int(min(cx2 + Wd / 2 + mx, rot.shape[1]))
    by0 = int(max(cy2 - L / 2 - my, 0)); by1 = int(min(cy2 + L / 2 + my, rot.shape[0]))
    rect = dict(cx=cx2 - bx0, cy=cy2 - by0, L=L, W=Wd)
    return rot[by0:by1, bx0:bx1], rect


# regulation-fixed controls, Laws of the Game (IFAB) Law 1
CONTROLS = [("Penalty area width", 40.32), ("Penalty area depth", 16.50),
            ("Goal area width", 18.32), ("Goal area depth", 5.50)]


def goal_end(panel, rect, m_per_px):
    """Which end carries the painted penalty box, and where its line actually is.

    The box line is a thin bright ridge 16.50 m from the goal line. Scoring each end
    by how well its strongest ridge agrees with that distance is far more reliable
    than raw brightness, which goal structures and worn turf both inflate.
    """
    ppm = 1.0 / m_per_px
    cx, cy, L = rect["cx"], rect["cy"], rect["L"]
    pw, pd = 40.32 * ppm / 2, 16.50 * ppm
    g = panel.mean(2).astype(float) if panel.ndim == 3 else panel.astype(float)
    loc = g - ndi.uniform_filter(g, 15)
    xs = slice(int(max(cx - 0.9 * pw, 0)), int(min(cx + 0.9 * pw, g.shape[1])))
    best = {}
    for end in (-1, 1):
        gl = cy + end * L / 2
        cand = []
        for d in range(int(0.5 * pd), int(1.6 * pd)):
            y = int(gl - end * d)
            if 0 <= y < g.shape[0]:
                w = np.exp(-((d - pd) / (0.18 * pd)) ** 2)
                cand.append((loc[y, xs].mean() * w, d, y))
        best[end] = max(cand) if cand else (-1e9, pd, None)
    end = 1 if best[1][0] >= best[-1][0] else -1
    score, d, y = best[end]
    return end, (cy + end * L / 2 if y is None else y + end * pd), score, d * m_per_px


def draw_baselines(ax, panel, rect, m_per_px, colour=None, halo=None, fs=6.0, end=None):
    """The four measured baselines, at true length, on the end that carries the box."""
    colour = colour or LINE_COLOUR
    halo = halo or HALO_COLOUR
    ppm = 1.0 / m_per_px
    cx, cy, L = rect["cx"], rect["cy"], rect["L"]
    if end is None:
        end, gl, sc, dm = goal_end(panel, rect, m_per_px)
        print("   goal end %s, penalty line found at %.2f m (true 16.50 m)"
              % ("bottom" if end > 0 else "top", dm))
    else:
        gl = cy + end * L / 2                   # the goal line
    inward = -end                               # towards the centre circle
    pw, pd = 40.32 * ppm / 2, 16.50 * ppm
    gw, gd = 18.32 * ppm / 2, 5.50 * ppm
    stroke = [pe.withStroke(linewidth=2.0, foreground=halo)]
    tstroke = [pe.withStroke(linewidth=1.6, foreground=halo)]

    def seg(x0, y0, x1, y1):
        ax.plot([x0, x1], [y0, y1], color=colour, lw=1.0, solid_capstyle="butt",
                marker="|" if abs(y0 - y1) < 1e-6 else "_", ms=3.5, mew=1.0,
                path_effects=stroke, zorder=4)

    def lab(x, y, t, ha="center", va="center", dx=0, dy=0):
        ax.text(x + dx, y + dy, t, color=colour, fontsize=fs, ha=ha, va=va,
                path_effects=tstroke, zorder=5)

    y_pen = gl + inward * pd
    y_goal = gl + inward * gd
    seg(cx - pw, y_pen, cx + pw, y_pen)                             # 40.32 m
    lab(cx, y_pen + inward * 4, "40.32 m", va="top" if inward > 0 else "bottom")
    seg(cx - pw, gl, cx - pw, y_pen)                                # 16.50 m
    lab(cx - pw - 4, (gl + y_pen) / 2, "16.50 m", ha="right")
    seg(cx - gw, y_goal, cx + gw, y_goal)                           # 18.32 m
    lab(cx, y_goal + inward * 4, "18.32 m", va="top" if inward > 0 else "bottom")
    seg(cx + gw, gl, cx + gw, y_goal)                               # 5.50 m
    lab(cx + gw + 4, (gl + y_goal) / 2, "5.50 m", ha="left")


def pitch_panel(ax):
    path = find_file(FIGA1_TILE, REAL_IMGS)
    img = load_rgb(path) if path else None
    if img is None:
        show(ax, None, "set FIGA1_TILE")
        return False
    if FIGA1_CROP is not None:
        sub = crop(img, FIGA1_CROP)
        rot = sub if FIGA1_ANGLE is None else np.array(
            Image.fromarray(sub).rotate(FIGA1_ANGLE, resample=Image.BICUBIC, expand=True))
        q = find_pitch(rot)
        rect = None if q is None else dict(cx=q["centre"][0], cy=q["centre"][1],
                                           L=max(q["length"], q["width"]),
                                           W=min(q["length"], q["width"]))
        panel = rot
    elif PITCH_AUTO:
        panel, rect = upright_pitch(img, FIGA1_ANGLE)
    else:
        panel, rect = img, None
    if panel is None:
        show(ax, None, "pitch not found -\nset FIGA1_CROP")
        return False
    show(ax, panel)
    if rect is not None:
        draw_baselines(ax, panel, rect, GSD_M_PER_PX)
        print("pitch: %d x %d px  ->  %.1f x %.1f m at %.4f m/px"
              % (rect["L"], rect["W"], rect["L"] * GSD_M_PER_PX,
                 rect["W"] * GSD_M_PER_PX, GSD_M_PER_PX))
    return True

## 6 · The six builders

In [ ]:
# =====================================================================================
#  The six figures. Each is a function; the last cell calls the ones you selected.
# =====================================================================================

GSD_CONTROLS = [("Penalty area width", 40.32, 0.2998), ("Penalty area depth", 16.50, 0.2989),
                ("Goal area width", 18.32, 0.3061), ("Goal area depth", 5.50, 0.3087)]
GSD_COMBINED, GSD_HALFRANGE = 0.3016, 0.0049

ROLE_SCORE = [
    lambda s: TILE_STATS[s]["Residential"] - 0.5 * TILE_STATS[s]["Forest"],
    lambda s: TILE_STATS[s]["River"] + 0.2 * TILE_STATS[s]["Residential"],
    lambda s: (TILE_STATS[s]["Agricultural"] + 0.6 * TILE_STATS[s]["UnusedLand"]
               - 0.8 * TILE_STATS[s]["Residential"]),
]


def auto_tiles(n=3, used=()):
    """Fill empty slots: a dense tile, a river tile, a cultivated tile."""
    out, taken = [], list(used)
    for i in range(n):
        f = ROLE_SCORE[i % len(ROLE_SCORE)]
        pool = [(f(s), s) for s in auto_stems() if s not in taken]
        pool = [p for p in pool if p[0] > 0]
        if not pool:
            out.append(None); continue
        s = max(pool)[1]
        taken.append(s); out.append(s)
    return out


def row_arrays(row, idx, auto):
    """One row of Fig. 1 / Fig. 9: image, annotation, and the stem behind them."""
    ip = find_file(row.get("image", ""), REAL_IMGS)
    if ip is None and auto is not None:
        ip = find_file(auto, REAL_IMGS)
    if ip is None:
        return None, None, None, None
    s = stem(ip)
    mp = find_file(row.get("mask", ""), REAL_MASKS) or mask_for(ip)
    img = load_rgb(ip)
    mk = load_mask(mp) if mp else None
    print("   row: %s  +  %s" % (Path(ip).name, Path(mp).name if mp else "NO MASK"))
    if mp:
        pair_ok(ip, mp)
    return img, mk, s, mp


# ------------------------------------------------------------------------- Fig. 1
def build_fig01():
    autos = auto_tiles(3)
    fig, axs = plt.subplots(3, 4, figsize=(W2, W2 * 3 / 4 * 1.02), layout="constrained")
    gs = axs[0, 0].get_gridspec()
    for a in axs[:, 0]:
        a.remove()
    axloc = fig.add_subplot(gs[:, 0])
    draw_locator(axloc)

    titles = ["image", "annotation", "overlay"]
    for r in range(3):
        row = FIG1_ROWS[r] if r < len(FIG1_ROWS) else {}
        img, mk, s, _ = row_arrays(row, r, autos[r])
        ov = load_rgb(row.get("overlay", "")) if row.get("overlay") else None
        if ov is None and row.get("overlay"):
            ov = load_rgb(find_file(row["overlay"], REAL_IMGS) or "")
        if ov is None:
            ov = overlay(img, mk)
        for c, arr in enumerate([img, mk, ov]):
            ax = axs[r, 1 + c]
            show(ax, arr, "set FIG1_ROWS[%d]['%s']" % (r, ["image", "mask", "overlay"][c]))
            if r == 0:
                ax.set_title(titles[c], fontsize=7, pad=2)
        lab = row.get("label") or (s or "")
        axs[r, 1].set_ylabel(lab, fontsize=6.5)

    class_legend(fig)
    tiles = [axs[r, c] for r in range(3) for c in range(1, 4)]
    finish(fig, "fig01_study_area_and_tiles", "Fig. 1", "Section 3.1",
           "at the end of the subsection", "figure*",
           "The corpus. (a) Location of the Kathmandu Valley study area within Nepal, with the "
           "boundary of the annotated extent shown beneath it; per-tile "
           "footprints were not preserved, so the extent is approximate and no georeferenced "
           "tile index is shown. Map lines delineate study areas and do not necessarily depict "
           "accepted national boundaries. (b) Three tiles as image, annotation and overlay. The "
           "palette shown is used in every subsequent figure.",
           axes=[axloc, tiles])
    plt.show()


# ------------------------------------------------------------------------- Fig. 6
TAU = (0.60, 0.35, 0.35, 0.60, 0.60, 0.60)     # CSA_TAU, in CLASSES order
W_MIN = 0.25
TILE_METRES, CACHE_PX = 300.0, 512
D_MAX_PX = 6.0 / (TILE_METRES / CACHE_PX)      # CSA_DMAX_M in ground metres -> pixels


def csa_weight(idx, taus=TAU, wmin=W_MIN, dmax_px=D_MAX_PX):
    """Eq. (1), ported from csa_weight() in the training notebook."""
    from scipy import ndimage as ndi
    lab = idx.astype(np.int16)
    b = np.zeros(lab.shape, bool)
    d = lab[:-1, :] != lab[1:, :]; b[:-1, :] |= d; b[1:, :] |= d
    d = lab[:, :-1] != lab[:, 1:]; b[:, :-1] |= d; b[:, 1:] |= d
    D = ndi.distance_transform_edt(~b).astype(np.float32)
    W = np.ones(lab.shape, np.float32)
    BAND = np.zeros(lab.shape, np.float32)                     # the band width per pixel
    st = np.ones((3, 3), int)                                  # 8-connectivity
    for c in range(len(CLASSES)):
        m = (idx == c)
        if not m.any():
            continue
        cc, n = ndi.label(m, structure=st)
        if n == 0:
            continue
        means = np.asarray(ndi.mean(D, labels=cc, index=np.arange(1, n + 1)), np.float32)
        band = np.maximum(np.minimum(taus[c] * 2.0 * means, dmax_px), 1.0)
        bmap = np.zeros(n + 1, np.float32); bmap[1:] = band
        W[m] = np.clip(D[m] / bmap[cc[m]], wmin, 1.0)
        BAND[m] = bmap[cc[m]]
    W[idx == 255] = 1.0
    return b, D, W, BAND


def pick_crop(idx, W, BAND, side=224):
    """The crop that SHOWS the mechanism, not just one that contains a road.

    Eq. (1) makes the band a fraction of local object width, capped at 10.24 px. A big
    residential block always sits at that cap, and a road three pixels wide is thinner
    than its own one-pixel band, so a crop of road-inside-residential shows two bands of
    identical width and demonstrates nothing. The window is therefore scored on how much
    the band width VARIES inside it - which is exactly the property the panel exists to
    show - and on how much of it is banded at all.
    """
    H, Wd = idx.shape
    best, box, why = -1, (0, 0, side, side), ""
    for y in range(0, max(H - side, 0) + 1, max(side // 4, 1)):
        for x in range(0, max(Wd - side, 0) + 1, max(side // 4, 1)):
            w = W[y:y + side, x:x + side]
            b = BAND[y:y + side, x:x + side]
            m = w < 0.999
            frac = float(m.mean())
            if frac < 0.02:
                continue
            lo, hi = np.percentile(b[m], 10), np.percentile(b[m], 90)
            score = frac ** 0.5 * (hi - lo + 0.5)
            if score > best:
                best, box = score, (x, y, x + side, y + side)
                why = ("%.0f%% of the crop is banded, band widths %.1f to %.1f px"
                       % (100 * frac, lo, hi))
    return box, why


def build_fig06():
    ip = find_file(FIG6_IMAGE, REAL_IMGS)          # the manual slots win outright
    mp = find_file(FIG6_MASK, REAL_MASKS)
    s6 = FIG6_TILE or ""
    if not s6 and not (ip or mp) and TILE_STATS:
        def demo_score(s, need_river):
            road = min(TILE_STATS[s]["Road"] / 0.03, 1.0)
            area = min(max(TILE_STATS[s]["Residential"],
                           TILE_STATS[s]["Agricultural"]) / 0.25, 1.0)
            riv = min(TILE_STATS[s]["River"] / 0.02, 1.0)
            return road * area * (riv if need_river else 1.0)
        # first choice: a tile with road AND river AND a large area class, so the panel
        # shows three different band widths rather than one repeated three times
        for need_river in (True, False):
            pool = [(demo_score(s, need_river), s) for s in auto_stems()]
            pool = [p for p in pool if p[0] > 0]
            if pool:
                s6 = max(pool)[1]
                break
    if s6:                                     # one stem fills whichever slot is empty
        ip = ip or find_file(s6, REAL_IMGS)
        mp = mp or find_file(s6, REAL_MASKS)
    if ip and not mp:
        mp = mask_for(ip)
    if mp and not ip:
        ip = image_for(mp)
    s6 = stem(ip) if ip else s6
    print("   using %s  +  %s" % (Path(ip).name if ip else "NO IMAGE",
                                  Path(mp).name if mp else "NO MASK"))
    if ip and mp:
        pair_ok(ip, mp)
    img = load_rgb(ip) if ip else None
    idx, cov = to_index(load_rgb(mp)) if mp else (None, 0)

    fig, axs = plt.subplots(1, 4, figsize=(W2, W2 / 4 * 1.16), layout="constrained")
    if img is None or idx is None:
        for ax in axs:
            show(ax, None, "set FIG6_TILE")
    else:
        # the weight field is what training computes: on the 512 cache, whole tile
        idx512 = np.array(Image.fromarray(idx.astype(np.uint8)).resize(
            (CACHE_PX, CACHE_PX), Image.NEAREST)).astype(np.int16)
        img512 = np.array(Image.fromarray(img).resize((CACHE_PX, CACHE_PX), Image.BILINEAR))
        B, D, Wt, BAND = csa_weight(idx512)
        if FIG6_CROP:
            box, why = FIG6_CROP, "set by hand"
        else:
            box, why = pick_crop(idx512, Wt, BAND)
        print("Fig. 6: tile %s, crop %s of the 512 cache, band capped at %.2f px (6.00 m)"
              % (s6, box, D_MAX_PX))
        print("        %s" % why)
        ann = paint(crop(idx512, box)); Bc = crop(B, box)
        ann[Bc] = 255
        show(axs[0], crop(img512, box))
        show(axs[1], ann)
        # D is clipped for display: unclipped, the interior of a large polygon (60+ px
        # from any boundary) takes the whole colour range and the band structure the
        # panel is about renders as flat black
        axs[2].imshow(crop(D, box), cmap="magma", interpolation="nearest",
                      vmin=0, vmax=3 * D_MAX_PX)
        im3 = axs[3].imshow(crop(Wt, box), cmap="viridis", vmin=W_MIN, vmax=1.0,
                            interpolation="nearest")
        for ax in axs[2:]:
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_linewidth(0.5); sp.set_color("0.6")
        cb = fig.colorbar(im3, ax=axs[3], fraction=0.046, pad=0.02)
        cb.set_label("$w(x)$", fontsize=7); cb.ax.tick_params(labelsize=6)
        cb.outline.set_linewidth(0.5)

    finish(fig, "fig06_reliability_weight", "Fig. 6", "Section 4.1",
           "immediately after Eq. (1) and the sentence giving its parameters", "figure*",
           "The per-pixel reliability weight of Eq. (1), computed on the 512-pixel working "
           "resolution for one crop containing both a road and a large area class. (a) Image. "
           "(b) Annotation with the boundary set B overdrawn in white. (c) Euclidean distance "
           "transform of the complement of B. (d) The resulting weight field, clipped to a floor "
           "of 0.25. Because the unreliable band is a fraction of local object width rather than "
           "a fixed number of pixels, it stays narrow on the road and widens on the area class.",
           axes=list(axs))
    plt.show()


# ------------------------------------------------------------------------- Fig. 9
def build_fig09():
    given = [r.get("image", "") for r in FIG9_ROWS]
    autos = [None, None, None]
    sc = scorer()
    if not any(given) and sc is not None and TILE_STATS:
        ordered = sorted(auto_stems(), key=lambda s: (int(_digits(s) or 0), s))
        n = len(ordered) // 3
        val = ordered[FOLD * n:(FOLD + 1) * n] if n else ordered
        val = [s for s in val if find_file(s, REAL_IMGS) and find_file(s, REAL_MASKS)]
        if len(val) > MAX_SCORE_SCENES:
            val = [val[i] for i in np.linspace(0, len(val) - 1, MAX_SCORE_SCENES).astype(int)]
        print("scoring", len(val), "validation scenes of fold", FOLD, "...")
        rank = []
        for s in val:
            img = load_rgb(find_file(s, REAL_IMGS))
            gt, cov = to_index(load_rgb(find_file(s, REAL_MASKS)))
            if img is None or gt is None or cov < 0.2:
                continue
            gt = np.array(Image.fromarray(gt.astype(np.uint8)).resize((512, 512), Image.NEAREST))
            v = iou_mean(predict(sc, img), gt)
            if not np.isnan(v):
                rank.append((v, s))
        if len(rank) >= 3:
            rank.sort()
            autos = [rank[-1][1], rank[len(rank) // 2][1], rank[0][1]]
            print("   best %.3f · median %.3f · worst %.3f mIoU"
                  % (rank[-1][0], rank[len(rank) // 2][0], rank[0][0]))
    elif not any(given):
        autos = auto_tiles(3)

    cols = ["image", "reference annotation", "SegFormer-B0", "proposed model"]
    fig, axs = plt.subplots(3, 4, figsize=(W2, W2 * 3 / 4 * 1.03), layout="constrained")
    for r in range(3):
        row = FIG9_ROWS[r] if r < len(FIG9_ROWS) else {}
        img, mk, s, mp = row_arrays(row, r, autos[r])
        preds = []
        for tag in ("segformer-b0", "tc-segformer"):
            if img is not None and tag in MODELS:
                preds.append(paint(predict(MODELS[tag], img)))
            else:
                preds.append(None)
        notes = ["set FIG9_ROWS[%d]['image']" % r, "annotation not found",
                 "SegFormer-B0 output\nnot available", "proposed-model output\nnot available"]
        for c, arr in enumerate([img, mk] + preds):
            show(axs[r, c], arr, notes[c])
            if r == 0:
                axs[r, c].set_title(cols[c], fontsize=7, pad=2)
        axs[r, 0].set_ylabel(row.get("label") or (s or ""), fontsize=6.5)

    class_legend(fig)
    finish(fig, "fig09_qualitative_comparison", "Fig. 9", "Section 6.2",
           "at the end of the subsection, after the sentence on the dominant error modes",
           "figure*",
           "Qualitative comparison on held-out scenes of block fold 1: image, reference "
           "annotation, the SegFormer-B0 baseline and the proposed model, both run from their "
           "published checkpoints. The dominant error modes are the ones the separability matrix "
           "predicts: vacant ground absorbed into cultivated ground on the valley fringe, urban "
           "tree cover claimed as forest, and river fragmenting where it narrows.")
    plt.show()


# ------------------------------------------------------------------------- Fig. 11
def _fake_pair(i):
    gi = find_file("genimage" + str(i), FAKE_IMGS)
    gm = find_file("genmask" + str(i), FAKE_MASKS)
    return gi, gm


def build_fig11():
    ids = [str(r.get("index", "")) for r in FIG11_ROWS]
    given = any(r.get("image") or r.get("mask") for r in FIG11_ROWS)
    sc = scorer()
    scores = [None, None, None]
    if not any(ids) and not given and FAKE_IMGS and FAKE_MASKS:
        pool = sorted({_digits(stem(p)) for p in FAKE_MASKS}, key=lambda t: int(t or 0))
        if len(pool) > MAX_SCORE_PAIRS:
            pool = [pool[i] for i in np.linspace(0, len(pool) - 1, MAX_SCORE_PAIRS).astype(int)]
        vals = []
        for i in pool:
            gi, gm = _fake_pair(i)
            if not gi or not gm:
                continue
            m, cov = to_index(load_rgb(gm))
            if m is None or cov < 0.2:
                continue
            riv = float((m == CLASSES.index("River")).mean())
            v = np.nan
            if sc is not None:
                img = load_rgb(gi, bgr=True)          # genimage*.png is BGR on disk
                m512 = np.array(Image.fromarray(m.astype(np.uint8)).resize((512, 512),
                                                                          Image.NEAREST))
                v = iou_mean(predict(sc, img), m512)
            vals.append((v, i, riv))
        good = [v for v in vals if not np.isnan(v[0])]
        if len(good) >= 3:
            good.sort(reverse=True)
            river = [v for v in good if v[2] > 0.005]
            rows = [good[0], good[len(good) // 2], (river[-1] if river else good[-1])]
            ids = [r[1] for r in rows]; scores = [r[0] for r in rows]
            print("Fig. 11 layout mIoU: %.3f / %.3f / %.3f" % tuple(scores))
        elif vals:
            vals.sort(key=lambda v: -v[2])
            ids = [v[1] for v in vals[:3]]

    cols = ["generated mask", "generated image", "frozen scorer read-back"]
    fig, axs = plt.subplots(3, 3, figsize=(W2, W2 * 0.80), layout="constrained")
    for r in range(3):
        row = FIG11_ROWS[r] if r < len(FIG11_ROWS) else {}
        gi = find_file(row.get("image", ""), FAKE_IMGS)      # manual slots first
        gm = find_file(row.get("mask", ""), FAKE_MASKS)
        i = str(row.get("index", "") or (ids[r] if r < len(ids) else ""))
        if (gi is None or gm is None) and i != "":
            a_, b_ = _fake_pair(i)
            gi = gi or a_; gm = gm or b_
        m, _ = to_index(load_rgb(gm)) if gm else (None, 0)
        gmask = paint(m) if m is not None else None
        gimg = load_rgb(gi, bgr=True) if gi else None       # cv2-written: BGR on disk
        rb = None
        if gimg is not None and sc is not None:
            pr = predict(sc, gimg)
            rb = paint(pr)
            if scores[r] is None and m is not None:
                m512 = np.array(Image.fromarray(m.astype(np.uint8)).resize((512, 512),
                                                                          Image.NEAREST))
                scores[r] = iou_mean(pr, m512)
        notes = ["set FIG11_ROWS[%d]['index']" % r, "generated image not found",
                 "scorer read-back\nnot available"]
        for c, arr in enumerate([gmask, gimg, rb]):
            show(axs[r, c], arr, notes[c])
            if r == 0:
                axs[r, c].set_title(cols[c], fontsize=7, pad=2)
        if scores[r] is not None and rb is not None:
            axs[r, 2].text(0.97, 0.04, "layout mIoU %.3f" % scores[r],
                           transform=axs[r, 2].transAxes, ha="right", va="bottom",
                           fontsize=6.0, color="white",
                           bbox=dict(fc="0.15", ec="none", pad=1.2, alpha=0.75))
        axs[r, 0].set_ylabel(row.get("label") or "", fontsize=6.5)

    class_legend(fig)
    finish(fig, "fig11_synthetic_pairs", "Fig. 11", "Section 6.4",
           "at the end of the subsection, after the presence-rate sentence", "figure*",
           "Three synthetic pairs from " + FAKE_SET.replace("_", " ") + " spanning the fidelity "
           "range: the mask drawn by the Stage-1 diffusion model, the image Stage 2 paints from "
           "it, and the frozen scorer's read-back of that image against the mask that generated "
           "it, with the per-pair layout intersection over union inset. The bottom row shows the "
           "failure that drives the downstream result: river arriving as a wide fragmented blob "
           "rather than a narrow continuous ribbon.")
    plt.show()


# ------------------------------------------------------------------------- Fig. A.1
def build_figA1():
    fig, axs = plt.subplots(2, 1, figsize=(W1, 118 * MM), layout="constrained",
                            gridspec_kw=dict(height_ratios=[1.35, 1.0]))
    pitch_panel(axs[0])

    ax = axs[1]
    names = [c[0] for c in GSD_CONTROLS]
    true_m = np.array([c[1] for c in GSD_CONTROLS])
    gsd = np.array([c[2] for c in GSD_CONTROLS])
    y = np.arange(len(names))[::-1]
    ax.axvspan(GSD_COMBINED - GSD_HALFRANGE, GSD_COMBINED + GSD_HALFRANGE,
               color="#2C7FB8", alpha=0.13, lw=0, zorder=0)
    ax.axvline(GSD_COMBINED, color="#2C7FB8", lw=1.0, zorder=1)
    for k in range(len(names)):
        ax.plot([GSD_COMBINED, gsd[k]], [y[k], y[k]], color="0.8", lw=0.7, zorder=1)
        ax.scatter(gsd[k], y[k], s=18 + 42 * (true_m[k] / true_m.max()), color="#B2182B",
                   ec="white", lw=0.6, zorder=3)
        ax.text(gsd[k], y[k] + 0.30, "%.4f" % gsd[k], ha="center", fontsize=6.3)
    ax.set_yticks(y)
    ax.set_yticklabels([n + " (" + ("%.2f" % m) + " m)" for n, m in zip(names, true_m)])
    ax.set_xlabel("ground sampling distance (m per pixel)")
    ax.set_xlim(0.2960, 0.3120); ax.set_ylim(-0.55, len(names) - 0.35)
    ax.grid(axis="x", ls=":", lw=0.4); ax.set_axisbelow(True)

    top_legend(fig, [Line2D([], [], color="#2C7FB8"), Patch(fc="#2C7FB8", alpha=0.13)],
               ["length-weighted estimate, 0.3016", "half-range, " + r"$\pm$" + "0.0049"], ncol=2)

    finish(fig, "figA1_scale_recovery", "Fig. A.1", "Appendix A",
           "immediately after Table A.1", "figure",
           "Recovering the acquisition scale. (a) The pitch after automatic location and "
           "rotation to upright, with the four regulation-fixed controls marked at the length "
           "they occupy at the recovered scale. Penalty-area and goal-area dimensions are fixed "
           "by the Laws of the Game independently of the size of the pitch, which is what makes "
           "them usable as a scale reference. (b) Ground sampling distance from those four "
           "controls, marker size proportional to baseline length; the two long baselines "
           "cluster and the two short ones scatter, which is what a constant per-line "
           "localisation error produces.",
           axes=[axs[0], axs[1]], hspace=0.06)
    plt.show()


# ------------------------------------------------------------- graphical abstract
def build_graphical_abstract():
    GA_W, GA_H = 1328 / 100, 531 / 100
    fig, axs = plt.subplots(1, 3, figsize=(GA_W, GA_H), layout="constrained")

    ax = axs[0]
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off")
    ax.text(0.5, 0.93, "A CHARACTERISED CORPUS", ha="center", fontsize=12,
            transform=ax.transAxes)
    ax.text(0.5, 0.06, "202.4 km$^2$  ·  408 scenes  ·  six classes\n"
            "every class pair inseparable in RGB", ha="center", fontsize=9,
            transform=ax.transAxes)
    gip = find_file(GA_IMAGE, REAL_IMGS) or (REAL_IMGS[0] if REAL_IMGS else None)
    gmp = find_file(GA_MASK, REAL_MASKS) or (find_file(stem(gip), REAL_MASKS) if gip else None)
    t_im = load_rgb(gip) if gip else None
    t_mk = load_mask(gmp) if gmp else None
    for arr, x0, note in ((t_im, 0.04, "set GA_IMAGE"), (t_mk, 0.52, "set GA_MASK")):
        sax = ax.inset_axes([x0, 0.26, 0.44, 0.58])
        show(sax, arr, note)

    ax = axs[1]
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off")
    ax.text(0.5, 0.93, "THE PARTITION MATTERS MORE\nTHAN THE ARCHITECTURE", ha="center",
            fontsize=12, transform=ax.transAxes)
    inner = ax.inset_axes([0.14, 0.24, 0.78, 0.48])
    archs = ["U-Net", "UNetFormer", "SegFormer-B0", "SegNeXt-T", "proposed"]
    blk = [0.3986, 0.4200, 0.3962, 0.4723, 0.4682]
    rnd = [0.4494, 0.5058, 0.4765, 0.5531, 0.5631]
    y = np.arange(5)[::-1]
    for k in range(5):
        inner.plot([blk[k], rnd[k]], [y[k], y[k]], color="0.75", lw=1.2)
        inner.plot(blk[k], y[k], "o", color="#B2182B", ms=5)
        inner.plot(rnd[k], y[k], "s", color="#2166AC", ms=4.5)
    inner.set_yticks(y); inner.set_yticklabels(archs, fontsize=8)
    inner.set_xlim(0.36, 0.60); inner.tick_params(labelsize=7)
    inner.set_xlabel("mean IoU", fontsize=8)
    inner.spines["top"].set_visible(False); inner.spines["right"].set_visible(False)
    ax.text(0.5, 0.06, "0.051 to 0.095 mIoU, every architecture,\nfor the split protocol alone",
            ha="center", fontsize=9, transform=ax.transAxes)

    ax = axs[2]
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off")
    ax.text(0.5, 0.93, "SYNTHETIC PAIRS DO NOT HELP", ha="center", fontsize=12,
            transform=ax.transAxes)
    inner = ax.inset_axes([0.16, 0.30, 0.74, 0.42])
    inner.barh([1, 0], [-0.0140, -0.0570], color=["#8C8C8C", "#1F6FB4"], height=0.5)
    inner.set_yticks([1, 0]); inner.set_yticklabels(["mean IoU", "river IoU"], fontsize=8)
    inner.axvline(0, color="0.2", lw=0.8)
    inner.set_xlim(-0.075, 0.012); inner.tick_params(labelsize=7)
    inner.set_xlabel("change from adding synthetic pairs", fontsize=8)
    inner.spines["top"].set_visible(False); inner.spines["right"].set_visible(False)
    ax.text(0.5, 0.06, "all five architectures lose, and the damage\n"
            "lands on the rare thin classes it was built to enrich", ha="center", fontsize=9,
            transform=ax.transAxes)

    fig.savefig(OUT / "graphical_abstract.pdf")
    fig.savefig(OUT / "graphical_abstract.png", dpi=600)
    REGISTER.append(dict(file="graphical_abstract.pdf", figure="Graphical abstract",
                         section="separate upload", insertion_point="not in the manuscript body",
                         float_env="-", size="133 x 53 mm",
                         caption="No caption. The graphical abstract stands alone."))
    print("graphical_abstract.pdf   1328 x 531 px minimum satisfied")
    plt.show()


print("6 builders defined")

## 7 · Selector, register and zip

In [ ]:
import csv, shutil, os, re, traceback

FIGURES = {
    "1":  ("Fig. 1  study area and tiles",      build_fig01),
    "6":  ("Fig. 6  reliability weight field",  build_fig06),
    "9":  ("Fig. 9  qualitative comparison",    build_fig09),
    "11": ("Fig. 11 synthetic pairs",           build_fig11),
    "A1": ("Fig. A.1 scale recovery (a)+(b)",   build_figA1),
    "GA": ("graphical abstract",                build_graphical_abstract),
}
ORDER = ["1", "6", "9", "11", "A1", "GA"]
COLS = ["file", "figure", "section", "insertion_point", "float_env", "size", "caption"]


def _select(choice):
    if choice is None or str(choice).strip().lower() in ("", "all", "*"):
        return list(ORDER)
    toks = [t.strip().upper() for t in re.split(r"[,\s]+", str(choice)) if t.strip()]
    out = [t for t in toks if t in FIGURES]
    bad = [t for t in toks if t not in FIGURES]
    if bad:
        print("ignored:", ", ".join(bad), "· valid keys:", ", ".join(ORDER))
    return out or list(ORDER)


def _register():
    rows = {}
    for r in REGISTER:
        rows[r["file"]] = r
    keep = [r for f, r in rows.items() if (OUT / f).exists()]
    with open(OUT / "figure_register.csv", "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.DictWriter(fh, fieldnames=COLS); w.writeheader()
        for r in keep:
            w.writerow(r)
    return keep


def _zip():
    dest = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    zp = shutil.make_archive(str(dest / "geonusaf_section3_figures"), "zip", root_dir=OUT)
    print("\nzip ->", zp, "(%.2f MB)" % (os.path.getsize(zp) / 1e6))
    try:
        from IPython.display import FileLink, display
        display(FileLink(os.path.relpath(zp, ".")))
    except Exception:
        pass
    return zp


def run(choice="all", zip_output=True):
    picked = _select(choice)
    print("=" * 86)
    print("BUILDING:", ", ".join(FIGURES[k][0] for k in picked))
    print("=" * 86)
    made, failed = [], []
    for k in [k for k in ORDER if k in picked]:
        title, fn = FIGURES[k]
        print("\n" + "-" * 86); print(title); print("-" * 86)
        try:
            fn(); made.append(k)
        except Exception:
            failed.append(k); traceback.print_exc()
            print("*** carrying on with the rest ***")
    keep = _register()
    print("\n" + "=" * 86)
    for r in keep:
        print("%-34s %-18s %-9s %s" % (r["file"], r["figure"], r["float_env"], r["size"]))
    print("-" * 86)
    print(len(made), "built ·", len(keep), "figures in", OUT.resolve())
    if failed:
        print("FAILED:", ", ".join(failed))
    if zip_output:
        _zip()
    return made


print("selector ready ·  run('all')  ·  run('1')  ·  run('9,11')  ·  keys:",
      ", ".join(ORDER))

---
## 8 · BUILD

In [ ]:
run(CHOICE)

---
## Notes

- **Fig. 9 and Fig. 11 need internet.** Kaggle: Session options → Internet → On. Without
  it the prediction panels state what is missing rather than showing something invented.
- **The pitch panel** finds the turf by colour, rotates the pitch upright, then snaps the
  40.32 m baseline to the painted penalty line it detects. On `image264` it locates that
  line at 16.59 m against a true 16.50 m, which is the check that it landed correctly.
  If your tile differs, set `FIGA1_CROP` and `FIGA1_ANGLE` by hand, or turn `PITCH_AUTO`
  off.
- **Overlays.** Leave `overlay` empty for the 50 % blend, or point it at your CVAT
  screenshot and that file is used as-is.
- **The study-area panel needs internet the first time** (Fig. 1 only). It reads the
  boundary from the `.shp` alone — no `.dbf`, no `.shx`, no geopandas — fetches the
  imagery tiles once, and caches the mosaic in `figs/_cache/valley_*.png`, so later runs
  are offline. With no internet it draws the boundary on a plain panel and says so.
  Google Earth's own tiles cannot be downloaded programmatically under its terms; to use
  a Google Earth view, screenshot it and set `LOCATOR_IMAGE`. Keep the imagery credit in
  the figure or the caption either way.
- **Sub-labels on stacked panels** share one x, so (a) and (b) of Fig. A.1 sit on one
  vertical line. `SUBLABEL_STACK_X` in section 1 takes `"union"`, `"mean"`, a figure
  fraction, or `None` for the old per-panel centring.
- The census is cached in `figs/_cache/tile_census.csv`; delete it after changing the
  dataset, or call `build_census(force=True)`.
- `pdfinfo figs/fig01_study_area_and_tiles.pdf` should report 539 pt (190 mm); Fig. A.1
  and anything single column report 255 pt (90 mm).
